In [ ]:
                    # Use the map function to add EUR values from the eur_cumulative_map DataFrame
                # Extract the XGBoost estimator from the pipeline
    # Extract numerical and categorical data from the filtered test set
#     #     # Extract the actual model from the pipeline if necessary
#     # Extract the SHAP values from the dictionary
# Suppress specific warnings from XGBoost
file_path = r'C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\Sample data from Vesna vLandedWells.xlsx'  
from IPython.display import display, HTML, IFrame
from PIL import Image
from folium import Map, Marker
from folium.plugins import Draw
from joblib import Parallel, delayed
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats, optimize
from shapely.geometry import LineString
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OneHotEncoder, PolynomialFeatures
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, Embedding, Flatten, Conv1D, MaxPooling1D, Reshape, GlobalAveragePooling1D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.regularizers import l1, l2
from textwrap import wrap
from tqdm import tqdm
from xgboost import XGBRegressor, DMatrix, train as xgb_train
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['HEELPOINT_LON'], df['HEELPOINT_LAT']))
import ast
import folium
import gc
import geopandas as gpd
import joblib
import logging
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
import shap
import tempfile
import time
import warnings
import xgboost as xgb

In [ ]:
warnings.filterwarnings(action='ignore', category=UserWarning, message='.*deprecated.*')

In [ ]:
warnings.filterwarnings(action='ignore', category=UserWarning, message='.*not used.*')

In [ ]:
warnings.filterwarnings(action='ignore', category=UserWarning, message='.*No visible GPU is found.*')

In [ ]:
display(HTML("""

In [ ]:
<style>

In [ ]:
.jp-OutputArea-output {

In [ ]:
    overflow-y: auto;

In [ ]:
}

In [ ]:
</style>

In [ ]:
"""))

In [ ]:
# Load the dataset

In [ ]:
df = pd.read_excel(file_path)

In [ ]:
# Read the first Excel sheet (cumulative data)

In [ ]:
file_path = r'C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\Cumulative Prod for sample data.xlsx'

In [ ]:
cumulative_df = pd.read_excel(file_path)

In [ ]:
df = pd.merge(df, cumulative_df, on='UWI10', how='left')

In [ ]:
# Identify rows in df that do not have a match in cumulative_df

In [ ]:
no_match_df = df[df['Cumulative oil mbo'].isnull() & 

In [ ]:
                        df['Cumulative gas mmcf'].isnull() & 

In [ ]:
                        df['Cumulative water mbbl'].isnull()]

In [ ]:
# Print the UWI10 values that did not have a match

In [ ]:
if no_match_df.empty:

In [ ]:
    print("All UWI10 values in the detailed well data have corresponding matches in the cumulative data.")

In [ ]:
else:

In [ ]:
    print("UWI10 values in the detailed well data that did not have a corresponding match in the cumulative data:")

In [ ]:
    print(no_match_df['UWI10'].tolist())

In [ ]:
    # Print the rows with no matching cumulative data

In [ ]:
    print("\nRows with no matching cumulative data:")

In [ ]:
    print(no_match_df)

In [ ]:
def robust_parse(x):

In [ ]:
    if pd.isna(x):

In [ ]:
        return [None] * 7  # Handle NaNs by returning a list of Nones

In [ ]:
    try:

In [ ]:
        if isinstance(x, str):

In [ ]:
            return ast.literal_eval(x)  # Attempt to parse string as a Python literal

In [ ]:
        else:

In [ ]:
            return [x] + [None] * 6  # If x is not a string, return it as the first element with None padding

In [ ]:
    except Exception as e:

In [ ]:
        #print(f"Failed to parse: {x} with error {e}")  # Optionally log the error for debugging

In [ ]:
        return [None] * 7  # Return None values if parsing fails

In [ ]:
# Columns to be parsed

In [ ]:
param_columns = [

In [ ]:
    'Oil_Params_P20', 'Gas_Params_P20', 'Oil_Params_P35', 'Gas_Params_P35', 

In [ ]:
    'Oil_Params_P50', 'Gas_Params_P50', 'Oil_Params_P65', 'Gas_Params_P65', 

In [ ]:
    'Oil_Params_P80', 'Gas_Params_P80', 'Water_Params_P50'

In [ ]:
]

In [ ]:
# Apply robust parsing to each column

In [ ]:
for col in param_columns:

In [ ]:
    df[col] = df[col].apply(robust_parse)

In [ ]:
# Split each parameter into its own column

In [ ]:
new_columns = []

In [ ]:
for col in param_columns:

In [ ]:
    expanded_cols = [f'{col}_Method', f'{col}_BuildupRate', f'{col}_MonthsInProd', 

In [ ]:
                     f'{col}_InitialProd', f'{col}_DiCoefficient', f'{col}_BCoefficient', 

In [ ]:
                     f'{col}_LimDeclineRate']

In [ ]:
    temp_df = pd.DataFrame(df[col].tolist(), columns=expanded_cols)

In [ ]:
    # Convert numeric columns to float

In [ ]:
    for num_col in expanded_cols:

In [ ]:
        if num_col.endswith('_Method'):

In [ ]:
            temp_df[num_col] = temp_df[num_col].astype(str)  # Ensuring 'Method' is of string type

In [ ]:
        else:

In [ ]:
            temp_df[num_col] = pd.to_numeric(temp_df[num_col], errors='coerce')  # Convert to numeric and handle errors

In [ ]:
    df = pd.concat([df, temp_df], axis=1)

In [ ]:
# Drop the original parameter columns

In [ ]:
df.drop(columns=param_columns, inplace=True)

In [ ]:
# Handling NaN values differently for categorical and numerical columns

In [ ]:
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
for column in df.columns:

In [ ]:
    if column in categorical_columns:

In [ ]:
        df[column] = df[column].fillna('Unknown')  # Using 'Unknown' for categorical data

In [ ]:
    else:

In [ ]:
        df[column] = df[column].fillna(0)  # Assuming 0 is a reasonable fill for numerical data

In [ ]:
# Drop rows where any of the new parameter columns have missing data

In [ ]:
df.dropna(subset=[col for col in df.columns if 'Params' in col], inplace=True)

In [ ]:
#df.columns

In [ ]:
dfnew= df[['UWI10','Typecurve']].drop_duplicates()

In [ ]:
dfnew.groupby('Typecurve',as_index=False).count()

In [ ]:
# Drop rows where 'FluidPerFoot_bblft' or 'ProppantPerFoot' are zero

In [ ]:
df = df[(df['FluidPerFoot_bblft'] != 0) & (df['ProppantPerFoot'] != 0)]

In [ ]:
df = df[(df['EUR_30yr_Actual_Gas_P50_MMCF'] != 0) & (df['EUR_30yr_Actual_Oil_P50_MBO'] != 0) & (df['EUR_30yr_Actual_Water_P50_MBBL'] != 0)]

In [ ]:
df = df[(df['HEELPOINT_LAT'] != 0)]

In [ ]:
# Define a function to replace zeros with the P50 value for the same category

In [ ]:
def replace_zeros_with_P50(df):

In [ ]:
    # Replace for EUR values

In [ ]:
    phases = ['Oil', 'Gas', 'Water']  # Assuming Water is also needed; adjust as necessary

In [ ]:
    years = ['30yr']  # Adjust or extend if there are other year ranges

In [ ]:
    for phase in phases:

In [ ]:
        for year in years:

In [ ]:
            p50_col = f'EUR_{year}_Actual_{phase}_P50_' + ('MBO' if phase != 'Gas' else 'MMCF')

In [ ]:
            if phase == 'Water':

In [ ]:
                p50_col = f'EUR_{year}_Actual_{phase}_P50_MBBL'  # Assuming water is measured in MBBL

In [ ]:
            for p in ['P20', 'P35', 'P65', 'P80']:

In [ ]:
                p_col = f'EUR_{year}_Actual_{phase}_{p}_' + ('MBO' if phase != 'Gas' else 'MMCF')

In [ ]:
                if phase == 'Water':

In [ ]:
                    p_col = f'EUR_{year}_Actual_{phase}_{p}_MBBL'

In [ ]:
                if p_col in df.columns and p50_col in df.columns:

In [ ]:
                    df.loc[df[p_col] == 0, p_col] = df[p50_col]

In [ ]:
    # Replace for parameters

In [ ]:
    params = ['Method', 'BuildupRate', 'MonthsInProd', 'InitialProd', 'DiCoefficient', 'BCoefficient', 'LimDeclineRate']

In [ ]:
    ###Log

In [ ]:
    for phase in ['Oil', 'Gas', 'Water']:  # Assuming Water parameters are also needed

In [ ]:
        for param in params:

In [ ]:
            p50_col = f'{phase}_Params_P50_{param}'

In [ ]:
            for p in ['P20', 'P35', 'P65', 'P80']:

In [ ]:
                p_col = f'{phase}_Params_{p}_{param}'

In [ ]:
                if p_col in df.columns and p50_col in df.columns:

In [ ]:
                    df.loc[df[p_col] == 0, p_col] = df[p50_col]

In [ ]:
replace_zeros_with_P50(df)

In [ ]:
# Drop rows where latitude or longitude values are missing

In [ ]:
df.dropna(subset=['HEELPOINT_LAT', 'HEELPOINT_LON', 'MIDPOINT_LAT', 'MIDPOINT_LON', 'TOEPOINT_LAT', 'TOEPOINT_LON'], inplace=True)

In [ ]:
# Create a GeoDataFrame

In [ ]:
# Create a LineString for each well

In [ ]:
gdf['line'] = gdf.apply(lambda row: LineString([

In [ ]:
    (row['HEELPOINT_LON'], row['HEELPOINT_LAT']),

In [ ]:
    (row['MIDPOINT_LON'], row['MIDPOINT_LAT']),

In [ ]:
    (row['TOEPOINT_LON'], row['TOEPOINT_LAT'])

In [ ]:
]), axis=1)

In [ ]:
# Create a GeoDataFrame with the LineStrings

In [ ]:
line_gdf = gpd.GeoDataFrame(gdf, geometry='line')

In [ ]:
# Calculate bounds to set the map's initial view

In [ ]:
bounds = line_gdf.total_bounds  # [minx, miny, maxx, maxy]

In [ ]:
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]  # [(miny + maxy)/2, (minx + maxx)/2]

In [ ]:
# Create a folium map centered on the calculated center

In [ ]:
m = folium.Map(location=center, zoom_start=10)

In [ ]:
# Fit map to bounds

In [ ]:
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

In [ ]:
# Add the lines to the map

In [ ]:
for _, row in line_gdf.iterrows():

In [ ]:
    line_points = [

In [ ]:
        (row['HEELPOINT_LAT'], row['HEELPOINT_LON']),

In [ ]:
        (row['MIDPOINT_LAT'], row['MIDPOINT_LON']),

In [ ]:
        (row['TOEPOINT_LAT'], row['TOEPOINT_LON'])

In [ ]:
    ]

In [ ]:
    folium.PolyLine(line_points, color='blue').add_to(m)

In [ ]:
    # Add marker for midpoint with a popup showing coordinates or other info

In [ ]:
    folium.Marker(location=line_points[1], popup=f'Well ID: {row["WellName"]}', tooltip='Click for info').add_to(m)

In [ ]:
# Add draw control to the map to allow for area selection

In [ ]:
draw = Draw(export=True)

In [ ]:
m.add_child(draw)

In [ ]:
# Display the map in the Jupyter notebook

In [ ]:
m.save('wells_map.html')

In [ ]:
display(IFrame('wells_map.html', width=700, height=500))

In [ ]:
def add_neighbor_eur_cumulative(df):

In [ ]:
    if df is None:

In [ ]:
        raise ValueError("The input DataFrame is None. Please provide a valid DataFrame.")

In [ ]:
    # Check if the UWI column exists

In [ ]:
    if 'UWI' not in df.columns:

In [ ]:
        raise ValueError("The input DataFrame does not contain the 'UWI' column.")

In [ ]:
    # Define the EUR columns we're interested in

In [ ]:
    eur_oil_columns = ['EUR_30yr_Actual_Oil_P20_MBO', 'EUR_30yr_Actual_Oil_P35_MBO', 

In [ ]:
                       'EUR_30yr_Actual_Oil_P50_MBO', 'EUR_30yr_Actual_Oil_P65_MBO', 

In [ ]:
                       'EUR_30yr_Actual_Oil_P80_MBO']

In [ ]:
    eur_gas_columns = ['EUR_30yr_Actual_Gas_P20_MMCF', 'EUR_30yr_Actual_Gas_P35_MMCF', 

In [ ]:
                       'EUR_30yr_Actual_Gas_P50_MMCF', 'EUR_30yr_Actual_Gas_P65_MMCF', 

In [ ]:
                       'EUR_30yr_Actual_Gas_P80_MMCF']

In [ ]:
    cumulative_columns = ['Cumulative oil mbo', 'Cumulative gas mmcf', 'Cumulative water mbbl']

In [ ]:
    # Check if the required EUR columns exist

In [ ]:
    missing_columns = [col for col in eur_oil_columns + eur_gas_columns + cumulative_columns if col not in df.columns]

In [ ]:
    if missing_columns:

In [ ]:
        raise ValueError(f"The input DataFrame is missing the following required columns: {missing_columns}")

In [ ]:
    # Create a mapping DataFrame that will be used for mapping EUR values

In [ ]:
    eur_cumulative_map = df.set_index('UWI')[eur_oil_columns + eur_gas_columns + cumulative_columns].fillna(0).copy()

In [ ]:
    # Create a dictionary to hold the new columns

In [ ]:
    new_columns = {}

In [ ]:
    # Iterate over NNAZ and NNSZ columns

In [ ]:
    for prefix in ['NNAZ', 'NNSZ']:

In [ ]:
        num_cols = 6 if prefix == 'NNAZ' else 2  # Assuming 6 NNAZ and 2 NNSZ columns

In [ ]:
        for i in range(1, num_cols + 1):

In [ ]:
            uwi_col = f'{prefix}_{i}_UWI'

In [ ]:
            if uwi_col in df.columns:  # Ensure the UWI column exists

In [ ]:
                for eur_col in eur_oil_columns + eur_gas_columns + cumulative_columns:

In [ ]:
                    new_col_name = f'{prefix}_{i}_{eur_col}'

In [ ]:
                    new_columns[new_col_name] = df[uwi_col].map(eur_cumulative_map[eur_col])

In [ ]:
    # Concatenate the new columns to the original dataframe

In [ ]:
    df = pd.concat([df, pd.DataFrame(new_columns)], axis=1)

In [ ]:
    return df

In [ ]:
df = add_neighbor_eur_cumulative(df)

In [ ]:
columns_to_drop=['UWI10', 'CompletionDate' ,'UWI', 'WellName','NNAZ_1_UWI','NNAZ_2_UWI','NNAZ_3_UWI','NNAZ_4_UWI','NNAZ_5_UWI','NNAZ_6_UWI','NNSZ_1_UWI',

In [ ]:
 'NNSZ_2_UWI','LeaseName', 'WellNumber', 'CurrentOperatorName', 'OriginalOperatorName', 'DrillingContractorName', 'PermitDate', 'SpudDate','FORMATION_CONDENSE', 'Unique_PDP_ID','EUR_30yr_Actual_Oil_P20_MBO',

In [ ]:
 'EUR_30yr_Actual_Gas_P20_MMCF', 'EUR_30yr_Actual_Oil_P35_MBO', 'EUR_30yr_Actual_Gas_P35_MMCF', 'EUR_30yr_Actual_Oil_P50_MBO', 'EUR_30yr_Actual_Gas_P50_MMCF',

In [ ]:
 'EUR_30yr_Actual_Oil_P65_MBO', 'EUR_30yr_Actual_Gas_P65_MMCF', 'EUR_30yr_Actual_Oil_P80_MBO', 'EUR_30yr_Actual_Gas_P80_MMCF', 'EUR_30yr_Actual_Water_P50_MBBL','WELL_TORTUOSITY','DEPTH_TO_TOP_2Q',

In [ ]:
 'DEPTH_TO_TOP_3Q', 'DEPTH_TO_TOP_4Q', 'AZIMUTH','DEPTH_ABOVE_ZONE_2Q',  'DEPTH_ABOVE_ZONE_3Q', 'DEPTH_ABOVE_ZONE_4Q','Cumulative oil mbo', 

In [ ]:
 'Cumulative gas mmcf', 'Cumulative water mbbl','AVERAGE_INCLINATION','DEPTH_TO_TOP_1Q','HEELPOINT_DEPTH','TOEPOINT_DEPTH',

In [ ]:
 'NNAZ_3_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P50_MBO',

In [ ]:
 'NNAZ_3_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_3_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_3_EUR_30yr_Actual_Gas_P20_MMCF',

In [ ]:
 'NNAZ_3_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_3_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_3_EUR_30yr_Actual_Gas_P65_MMCF',

In [ ]:
 'NNAZ_3_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_3_Cumulative oil mbo', 'NNAZ_3_Cumulative gas mmcf', 'NNAZ_3_Cumulative water mbbl',

In [ ]:
 'NNAZ_4_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P50_MBO',

In [ ]:
 'NNAZ_4_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_4_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_4_EUR_30yr_Actual_Gas_P20_MMCF',

In [ ]:
 'NNAZ_4_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_4_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_4_EUR_30yr_Actual_Gas_P65_MMCF',

In [ ]:
 'NNAZ_4_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_4_Cumulative oil mbo', 'NNAZ_4_Cumulative gas mmcf',

In [ ]:
 'NNAZ_4_Cumulative water mbbl', 'NNAZ_5_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P35_MBO',

In [ ]:
 'NNAZ_5_EUR_30yr_Actual_Oil_P50_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_5_EUR_30yr_Actual_Oil_P80_MBO',

In [ ]:
 'NNAZ_5_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P50_MMCF',

In [ ]:
 'NNAZ_5_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_5_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_5_Cumulative oil mbo',

In [ ]:
 'NNAZ_5_Cumulative gas mmcf', 'NNAZ_5_Cumulative water mbbl', 'NNAZ_6_EUR_30yr_Actual_Oil_P20_MBO',

In [ ]:
 'NNAZ_6_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_6_EUR_30yr_Actual_Oil_P50_MBO', 'NNAZ_6_EUR_30yr_Actual_Oil_P65_MBO',

In [ ]:
 'NNAZ_6_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_6_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P35_MMCF',

In [ ]:
 'NNAZ_6_EUR_30yr_Actual_Gas_P50_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_6_EUR_30yr_Actual_Gas_P80_MMCF',

In [ ]:
 'NNAZ_6_Cumulative oil mbo', 'NNAZ_6_Cumulative gas mmcf', 'NNAZ_6_Cumulative water mbbl','NNSZ_1_FORMATION', 'NNSZ_2_FORMATION','DEPTH_ABOVE_ZONE_1Q',

In [ ]:
 'NNAZ_1_FORMATION', 'NNAZ_2_FORMATION', 'NNAZ_3_FORMATION', 'NNAZ_4_FORMATION', 'NNAZ_5_FORMATION', 'NNAZ_6_FORMATION',

In [ ]:
 'NNAZ_1_EUR_30yr_Actual_Oil_P20_MBO', 'NNAZ_1_EUR_30yr_Actual_Oil_P35_MBO',  'NNAZ_1_EUR_30yr_Actual_Oil_P65_MBO',

In [ ]:
 'NNAZ_1_EUR_30yr_Actual_Oil_P80_MBO', 'NNAZ_1_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_1_EUR_30yr_Actual_Gas_P35_MMCF',

In [ ]:
 'NNAZ_1_EUR_30yr_Actual_Gas_P65_MMCF', 'NNAZ_1_EUR_30yr_Actual_Gas_P80_MMCF', 'NNAZ_2_EUR_30yr_Actual_Oil_P20_MBO',

In [ ]:
 'NNAZ_2_EUR_30yr_Actual_Oil_P35_MBO', 'NNAZ_2_EUR_30yr_Actual_Oil_P65_MBO', 'NNAZ_2_EUR_30yr_Actual_Oil_P80_MBO',

In [ ]:
 'NNAZ_2_EUR_30yr_Actual_Gas_P20_MMCF', 'NNAZ_2_EUR_30yr_Actual_Gas_P35_MMCF', 'NNAZ_2_EUR_30yr_Actual_Gas_P65_MMCF',

In [ ]:
 'NNAZ_2_EUR_30yr_Actual_Gas_P80_MMCF', 'NNSZ_1_EUR_30yr_Actual_Oil_P20_MBO', 'NNSZ_1_EUR_30yr_Actual_Oil_P35_MBO',

In [ ]:
 'NNSZ_1_EUR_30yr_Actual_Oil_P65_MBO', 'NNSZ_1_EUR_30yr_Actual_Oil_P80_MBO', 'NNSZ_1_EUR_30yr_Actual_Gas_P20_MMCF',

In [ ]:
 'NNSZ_1_EUR_30yr_Actual_Gas_P35_MMCF', 'NNSZ_1_EUR_30yr_Actual_Gas_P65_MMCF', 'NNSZ_1_EUR_30yr_Actual_Gas_P80_MMCF',

In [ ]:
 'NNSZ_2_EUR_30yr_Actual_Oil_P20_MBO', 'NNSZ_2_EUR_30yr_Actual_Oil_P35_MBO', 'NNSZ_2_EUR_30yr_Actual_Oil_P65_MBO',

In [ ]:
 'NNSZ_2_EUR_30yr_Actual_Oil_P80_MBO', 'NNSZ_2_EUR_30yr_Actual_Gas_P20_MMCF', 'NNSZ_2_EUR_30yr_Actual_Gas_P35_MMCF',

In [ ]:
 'NNSZ_2_EUR_30yr_Actual_Gas_P65_MMCF', 'NNSZ_2_EUR_30yr_Actual_Gas_P80_MMCF',

In [ ]:
]

In [ ]:
# Dropping the columns

In [ ]:
df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
#months_in_prod_columns = [col for col in df.columns if '_MonthsInProd' in col]

In [ ]:
#df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if '_Method' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
#months_in_prod_columns = [col for col in df.columns if 'LimDeclineRate' in col]

In [ ]:
#df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
# non_null_counts = df.count()

In [ ]:
# # Display columns with discrepancies in their count of non-null values

In [ ]:
# # This assumes the DataFrame has rows where some columns may be consistently non-null

In [ ]:
# max_count = non_null_counts.max()

In [ ]:
# discrepancy_columns = non_null_counts[non_null_counts < max_count]

In [ ]:
# discrepancy_columns

In [ ]:
# Fill missing values with zero

In [ ]:
df = df.fillna(0)

In [ ]:
non_null_counts = df.count()

In [ ]:
# Display columns with discrepancies in their count of non-null values

In [ ]:
# This assumes the DataFrame has rows where some columns may be consistently non-null

In [ ]:
max_count = non_null_counts.max()

In [ ]:
discrepancy_columns = non_null_counts[non_null_counts < max_count]

In [ ]:
discrepancy_columns

In [ ]:
# Define bounds for each basin in a dictionary.

In [ ]:
# The keys are basin names, and the values are tuples of (lat_min, lat_max, lon_min, lon_max).

In [ ]:
basin_bounds = {

In [ ]:
    'Midland': {'lat_range': (29, 34), 'lon_range': (-110, -109)}

In [ ]:
    # Add more basins with their geographic bounds 

In [ ]:
}

In [ ]:
def assign_basin_tc(row):

In [ ]:
    # Check if BasinTC is 0 or missing (use pd.isna() and explicitly handle pd.NA)

In [ ]:
    if pd.isna(row['BasinTC'])== 'Unknown' or row['BasinTC'] == 0:

In [ ]:
        matched_basin = None

In [ ]:
        for basin, bounds in basin_bounds.items():

In [ ]:
            # Ensure all comparisons are done within bounds

In [ ]:
            if ((bounds['lat_range'][0] <= row['HEELPOINT_LAT'] <= bounds['lat_range'][1] and

In [ ]:
                 bounds['lon_range'][0] <= row['HEELPOINT_LON'] <= bounds['lon_range'][1]) or

In [ ]:
                (bounds['lat_range'][0] <= row['MIDPOINT_LAT'] <= bounds['lat_range'][1] and

In [ ]:
                 bounds['lon_range'][0] <= row['MIDPOINT_LON'] <= bounds['lon_range'][1]) or

In [ ]:
                (bounds['lat_range'][0] <= row['TOEPOINT_LAT'] <= bounds['lat_range'][1] and

In [ ]:
                 bounds['lon_range'][0] <= row['TOEPOINT_LON'] <= bounds['lon_range'][1])):

In [ ]:
                matched_basin = basin

In [ ]:
                break

In [ ]:
        return matched_basin if matched_basin else row['BasinTC']

In [ ]:
    return row['BasinTC']

In [ ]:
# Apply the function to each row

In [ ]:
df['BasinTC'] = df.apply(assign_basin_tc, axis=1)

In [ ]:
# Drop rows where 'BasinTC' is 'Unknown'

In [ ]:
df = df[df['BasinTC'] != 'Unknown']

In [ ]:
months_in_prod_columns = [col for col in df.columns if '_LAT' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if '_LON' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if 'COMPLETION_RELATIONSHIP' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if 'WELL_TRAJECTORY' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if 'PRIMARY_FORMATION' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
months_in_prod_columns = [col for col in df.columns if 'Typecurve' in col]

In [ ]:
df.drop(months_in_prod_columns, axis=1, inplace=True)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df_orig = df

In [ ]:
# Print the shape of the DataFrame to confirm rows have been dropped

In [ ]:
print(f"Updated DataFrame shape: {df.shape}")

In [ ]:
# Ensure BasinTC and FORMATION_CONDENSED are categorical

In [ ]:
df['BasinTC'] = df['BasinTC'].astype(str)

In [ ]:
df['FORMATION_CONDENSED'] = df['FORMATION_CONDENSED'].astype(str)

In [ ]:
# Define categorical and numerical columns excluding 'BasinTC' and 'FORMATION_CONDENSED'

In [ ]:
categorical_columns = [col for col in df.select_dtypes(include=['object', 'category']).columns.tolist() if col not in ['BasinTC', 'FORMATION_CONDENSED']]

In [ ]:
# Prepare data for modeling

In [ ]:
y_headers = [col for col in df.columns if 'Params' in col]             # Define target headers dynamically if needed

In [ ]:
numerical_columns = [col for col in df.columns if col not in categorical_columns + ['BasinTC', 'FORMATION_CONDENSED', *y_headers]]

In [ ]:
# Update feature_columns excluding 'BasinTC' and 'FORMATION_CONDENSED'

In [ ]:
feature_columns = numerical_columns + categorical_columns

In [ ]:
list(y_headers)

In [ ]:
list(numerical_columns)

In [ ]:
list(categorical_columns)

In [ ]:
# # Calculate the number of unique categories and their names for each column

In [ ]:
# category_info = {col: {'count': df[col].nunique(), 'names': df[col].unique()} for col in categorical_columns}

In [ ]:
# # Print the number of categories grouped by the header name

In [ ]:
# for header, info in category_info.items():

In [ ]:
#     print(f"{header}:")

In [ ]:
#     print(f"  Count of Categories: {info['count']}")

In [ ]:
#     print(f"  Categories: {info['names']}\n")

In [ ]:
# # Print the shape of the DataFrame to confirm rows have been dropped

In [ ]:
# print(f"Updated DataFrame shape: {df.shape}")

In [ ]:
# Initialize a dictionary to keep LabelEncoders for each column

In [ ]:
encoders = {}

In [ ]:
# Encode categorical columns and store the encoders

In [ ]:
for col in categorical_columns:

In [ ]:
    le = LabelEncoder()

In [ ]:
    df[col] = le.fit_transform(df[col].astype(str))  # Convert and encode

In [ ]:
    encoders[col] = le  # Store the encoder for inverse_transform

In [ ]:
# Split data

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

In [ ]:
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

In [ ]:
# Define function to prepare data

In [ ]:
def remove_outliers(df, columns, threshold=3):

In [ ]:
    for col in columns:

In [ ]:
        df = df[(np.abs(stats.zscore(df[col])) < threshold)]

In [ ]:
    return df

In [ ]:
def prepare_data(df, numerical_columns, categorical_columns):

In [ ]:
    df = df.copy()  # Create a copy of the dataframe to avoid SettingWithCopyWarning

In [ ]:
    df.loc[:, categorical_columns] = df[categorical_columns].astype(np.int32)

In [ ]:
    return df

In [ ]:
# Define function to filter data by basin and formation

In [ ]:
def filter_by_basin_and_formation(dfinput, basin, formation):

In [ ]:
    return dfinput[(dfinput['BasinTC'] == basin) & (dfinput['FORMATION_CONDENSED'] == formation)]

In [ ]:
def debug_shapes(train, val, test, y_headers):

In [ ]:
    print(f'Train X shape: {train.shape}, Train Y shape: {train[y_headers].shape}')

In [ ]:
    print(f'Val X shape: {val.shape}, Val Y shape: {val[y_headers].shape}')

In [ ]:
    print(f'Test X shape: {test.shape}, Test Y shape: {test[y_headers].shape}')

In [ ]:
# Apply scaling on train, validate and test sets

In [ ]:
#train_df = remove_outliers(train_df, numerical_columns)

In [ ]:
# Fit separate scalers for input features and output parameters

In [ ]:
# input_scaler = StandardScaler()

In [ ]:
# output_scaler = StandardScaler()

In [ ]:
input_scaler = RobustScaler()

In [ ]:
output_scaler = RobustScaler()

In [ ]:
# Apply log transformation to BuildupRate and InitialProd columns before scaling

In [ ]:
log_transform_columns = [col for col in y_headers if 'BuildupRate' in col or 'InitialProd' in col]

In [ ]:
#log_transform_columns = [col for col in y_headers if 'MonthsInProd' in col or 'InitialProd' in col]

In [ ]:
#log_transform_columns = [col for col in y_headers if 'InitialProd' in col]

In [ ]:
#log_transform_columns = []

In [ ]:
# Replace negative values with zeros before log transformation

In [ ]:
train_df[log_transform_columns] = train_df[log_transform_columns].apply(lambda x: x.clip(lower=0))

In [ ]:
val_df[log_transform_columns] = val_df[log_transform_columns].apply(lambda x: x.clip(lower=0))

In [ ]:
test_df[log_transform_columns] = test_df[log_transform_columns].apply(lambda x: x.clip(lower=0))

In [ ]:
train_df[log_transform_columns] = np.log1p(train_df[log_transform_columns])

In [ ]:
val_df[log_transform_columns] = np.log1p(val_df[log_transform_columns])

In [ ]:
test_df[log_transform_columns] = np.log1p(test_df[log_transform_columns])

In [ ]:
input_scaler.fit(train_df[numerical_columns])

In [ ]:
output_scaler.fit(train_df[y_headers])

In [ ]:
# Prepare data by scaling numerical columns and encoding categorical columns

In [ ]:
train_df[numerical_columns] = input_scaler.transform(train_df[numerical_columns])

In [ ]:
train_df[y_headers] = output_scaler.transform(train_df[y_headers])

In [ ]:
val_df[numerical_columns] = input_scaler.transform(val_df[numerical_columns])

In [ ]:
val_df[y_headers] = output_scaler.transform(val_df[y_headers])

In [ ]:
test_df[numerical_columns] = input_scaler.transform(test_df[numerical_columns])

In [ ]:
test_df[y_headers] = output_scaler.transform(test_df[y_headers])

In [ ]:
len(train_df)

In [ ]:
# Assuming df, train_df, val_df, test_df are defined and contain the necessary columns

In [ ]:
unique_combinations = df[['BasinTC', 'FORMATION_CONDENSED']].drop_duplicates()

In [ ]:
class RealTimePlottingCallback(Callback):

In [ ]:
    def __init__(self, combo_description):

In [ ]:
        super().__init__()

In [ ]:
        self.combo_description = combo_description

In [ ]:
        self.epochs = []

In [ ]:
        self.losses = []

In [ ]:
        self.val_losses = []

In [ ]:
    def on_epoch_end(self, epoch, logs=None):

In [ ]:
        self.epochs.append(epoch)

In [ ]:
        self.losses.append(logs.get('loss'))

In [ ]:
        self.val_losses.append(logs.get('val_loss'))

In [ ]:
    def on_train_end(self, logs=None):

In [ ]:
        plt.figure(figsize=(10, 4))

In [ ]:
        plt.plot(self.epochs, self.losses, label='Training Loss', color='blue')

In [ ]:
        plt.plot(self.epochs, self.val_losses, label='Validation Loss', color='red')

In [ ]:
        plt.title(f'Training and Validation Loss for {self.combo_description}')

In [ ]:
        plt.xlabel('Epoch')

In [ ]:
        plt.ylabel('Loss')

In [ ]:
        plt.legend()

In [ ]:
        plt.grid(True)

In [ ]:
        plt.show()

In [ ]:
# # Check for GPU availability and set XGBoost model configuration

In [ ]:
# def get_xgb_regressor():

In [ ]:
#     # try:

In [ ]:
#     #     xgb_regressor = xgb.XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', gpu_id=0)

In [ ]:
#     #     xgb_regressor.fit(pd.DataFrame([[0]]), pd.DataFrame([0]))  # Attempt to fit with dummy data

In [ ]:
#     #     return xgb_regressor

In [ ]:
#     # except xgb.core.XGBoostError:

In [ ]:
#         return xgb.XGBRegressor(tree_method='hist')

In [ ]:
def plot_model_performance(history, title):

In [ ]:
    plt.figure(figsize=(12, 6))

In [ ]:
    plt.plot(history.history['loss'], label='Training Loss')

In [ ]:
    plt.plot(history.history['val_loss'], label='Validation Loss')

In [ ]:
    plt.title(title)

In [ ]:
    plt.xlabel('Epochs')

In [ ]:
    plt.ylabel('Loss')

In [ ]:
    plt.legend()

In [ ]:
    plt.grid(True)

In [ ]:
    plt.show()

In [ ]:
def plot_ml_performance(y_true, y_pred, title):

In [ ]:
    mse = mean_squared_error(y_true, y_pred)

In [ ]:
    mae = mean_absolute_error(y_true, y_pred)

In [ ]:
    r2 = r2_score(y_true, y_pred)

In [ ]:
    print(f"{title} - MSE: {mse}, MAE: {mae}, R²: {r2}")

In [ ]:
    plt.figure(figsize=(12, 6))

In [ ]:
    plt.scatter(y_true, y_pred, alpha=0.5)

In [ ]:
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')

In [ ]:
    plt.title(title)

In [ ]:
    plt.xlabel('True Values')

In [ ]:
    plt.ylabel('Predicted Values')

In [ ]:
    plt.grid(True)

In [ ]:
    plt.show()

In [ ]:
# Function to build the model

In [ ]:
def build_model(numerical_columns, categorical_columns, df, output_size, model_type='neural_network', **kwargs):

In [ ]:
    if model_type == 'neural_network':

In [ ]:
        embedding_output_dim = kwargs.get('embedding_output_dim', 20)

In [ ]:
        dense_layer_sizes = kwargs.get('dense_layer_sizes', [256, 128, 64, 32])

In [ ]:
        dropout_rate = kwargs.get('dropout_rate', 0.3)

In [ ]:
        regularization = kwargs.get('regularization', None)

In [ ]:
        activation = kwargs.get('activation', 'relu')

In [ ]:
        optimizer = kwargs.get('optimizer', 'adam')

In [ ]:
        loss_function = kwargs.get('loss_function', 'mse')

In [ ]:
        # numerical_input = Input(shape=(len(numerical_columns),), name='num_input')

In [ ]:
        # categorical_inputs = [Input(shape=(1,), name=f'cat_input_{i}') for i, _ in enumerate(categorical_columns)]

In [ ]:
        # embeddings = [Embedding(input_dim=df[col].nunique() + 1, output_dim=embedding_output_dim, name=f'emb_{col}')(cat_input)

In [ ]:
        #               for cat_input, col in zip(categorical_inputs, categorical_columns)]

In [ ]:
        # flat_embeddings = [Flatten()(emb) for emb in embeddings]

In [ ]:
        # merged = Concatenate()([numerical_input] + flat_embeddings)

In [ ]:
        # x = merged

In [ ]:
        # for size in dense_layer_sizes:

In [ ]:
        #     x = Dense(size, activation=activation, kernel_regularizer=regularization)(x)

In [ ]:
        #     x = Dropout(dropout_rate)(x)

In [ ]:
        # output = Dense(output_size, activation='linear')(x)  # Single output layer for all targets

In [ ]:
        # model = Model(inputs=[numerical_input] + categorical_inputs, outputs=output)

In [ ]:
                # Numerical input and reshaping for Conv1D layer

In [ ]:
        numerical_input = Input(shape=(len(numerical_columns),), name='num_input')

In [ ]:
        reshaped_numerical_input = Reshape((len(numerical_columns), 1))(numerical_input)

In [ ]:
        # Convolutional layers

In [ ]:
        x = Conv1D(filters=32, kernel_size=3, activation=activation, padding='same')(reshaped_numerical_input)

In [ ]:
        x = MaxPooling1D(pool_size=2)(x)

In [ ]:
        x = Conv1D(filters=64, kernel_size=3, activation=activation, padding='same')(x)

In [ ]:
        x = MaxPooling1D(pool_size=2)(x)

In [ ]:
        x = Conv1D(filters=128, kernel_size=3, activation=activation, padding='same')(x)

In [ ]:
        x = GlobalAveragePooling1D()(x)

In [ ]:
        # Categorical input and embeddings

In [ ]:
        categorical_inputs = [Input(shape=(1,), name=f'cat_input_{i}') for i, _ in enumerate(categorical_columns)]

In [ ]:
        embeddings = [Embedding(input_dim=df[col].nunique() + 1, output_dim=embedding_output_dim, name=f'emb_{col}')(cat_input)

In [ ]:
                      for cat_input, col in zip(categorical_inputs, categorical_columns)]

In [ ]:
        flat_embeddings = [Flatten()(emb) for emb in embeddings]

In [ ]:
        # Merging numerical and categorical inputs

In [ ]:
        merged = Concatenate()([x] + flat_embeddings)

In [ ]:
        # Dense layers

In [ ]:
        for size in dense_layer_sizes:

In [ ]:
            merged = Dense(size, activation=activation, kernel_regularizer=regularization)(merged)

In [ ]:
            merged = Dropout(dropout_rate)(merged)

In [ ]:
        output = Dense(output_size, activation='linear')(merged)  # Single output layer for all targets

In [ ]:
        model = Model(inputs=[numerical_input] + categorical_inputs, outputs=output)

In [ ]:
        if optimizer == 'adam':

In [ ]:
            opt = Adam(learning_rate=0.001)

In [ ]:
        elif optimizer == 'sgd':

In [ ]:
            opt = SGD(learning_rate=0.001)

In [ ]:
        elif optimizer == 'rmsprop':

In [ ]:
            opt = RMSprop(learning_rate=0.001)

In [ ]:
        model.compile(optimizer=opt, loss=loss_function)

In [ ]:
        return model

In [ ]:
    elif model_type in ['random_forest', 'decision_tree', 'xgboost']:

In [ ]:
        if model_type == 'random_forest':

In [ ]:
            model = RandomForestRegressor(n_jobs=-1, n_estimators=kwargs.get('n_estimators', 100), max_depth=kwargs.get('max_depth', 10), min_samples_split=kwargs.get('min_samples_split', 2), min_samples_leaf=kwargs.get('min_samples_leaf', 1), bootstrap=kwargs.get('bootstrap', True))

In [ ]:
        elif model_type == 'decision_tree':

In [ ]:
            model = DecisionTreeRegressor(max_depth=kwargs.get('max_depth', 20), min_samples_split=kwargs.get('min_samples_split', 5), min_samples_leaf=kwargs.get('min_samples_leaf', 2))

In [ ]:
        elif model_type == 'xgboost':

In [ ]:
            model = XGBRegressor(tree_method='hist',

In [ ]:
                                 n_estimators=kwargs.get('n_estimators', 200),

In [ ]:
                                 max_depth=kwargs.get('max_depth', 7),

In [ ]:
                                 learning_rate=kwargs.get('learning_rate', 0.01),

In [ ]:
                                 subsample=kwargs.get('subsample', 0.8),

In [ ]:
                                 colsample_bytree=kwargs.get('colsample_bytree', 0.8),

In [ ]:
                                 reg_alpha=kwargs.get('reg_alpha', 0.01), reg_lambda=kwargs.get('reg_lambda', 1), gamma=kwargs.get('gamma', 0.1))

In [ ]:
        # Preprocessing pipeline

In [ ]:
        numerical_transformer = Pipeline(steps=[

In [ ]:
            ('poly', PolynomialFeatures(degree=2, include_bias=False))

In [ ]:
        ])

In [ ]:
        categorical_transformer = Pipeline(steps=[

In [ ]:
            ('onehot', OneHotEncoder(handle_unknown='ignore'))

In [ ]:
        ])

In [ ]:
        preprocessor = ColumnTransformer(

In [ ]:
            transformers=[

In [ ]:
                ('num', numerical_transformer, numerical_columns),

In [ ]:
                ('cat', categorical_transformer, categorical_columns)

In [ ]:
            ]

In [ ]:
        )

In [ ]:
        pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

In [ ]:
        return pipeline

In [ ]:
# Define a helper function to format configuration dictionaries as strings

In [ ]:
def format_config(config):

In [ ]:
    formatted_config = ", ".join([f"{k}: {v}" for k, v in config.items() if not isinstance(v, object)])

In [ ]:
    return f"{{{formatted_config}}}"

In [ ]:
def train_and_evaluate_model(combo_train, combo_val, combo_test, numerical_columns, categorical_columns, y_headers, output_size, config, model_type, df, task_times):

In [ ]:
    start_time = time.time()

In [ ]:
    config_str = format_config(config)

In [ ]:
    print(f"Starting {model_type} training with config: {config_str}")

In [ ]:
    combo_train = prepare_data(combo_train, numerical_columns, categorical_columns)

In [ ]:
    combo_val = prepare_data(combo_val, numerical_columns, categorical_columns)

In [ ]:
    combo_test = prepare_data(combo_test, numerical_columns, categorical_columns)

In [ ]:
    if model_type == 'neural_network':

In [ ]:
        model = build_model(numerical_columns, categorical_columns, df, output_size, model_type=model_type, **config)

In [ ]:
        #model = build_cnn_model(numerical_columns, categorical_columns, df, output_size, **config)

In [ ]:
        real_time_plotter = RealTimePlottingCallback(combo_description=f'{config_str} - {model_type}')

In [ ]:
        history = model.fit(

In [ ]:
            x=[combo_train[numerical_columns].values] + [combo_train[col].astype(int).values.reshape(-1, 1) for col in categorical_columns], 

In [ ]:
            y=combo_train[y_headers].values,  # Use entire y_headers as output

In [ ]:
            validation_data=(

In [ ]:
                [combo_val[numerical_columns].values] + [combo_val[col].astype(int).values.reshape(-1, 1) for col in categorical_columns],

In [ ]:
                combo_val[y_headers].values  # Use entire y_headers as output

In [ ]:
            ),

In [ ]:
            epochs=1000, 

In [ ]:
            batch_size=50, 

In [ ]:
            callbacks=[

In [ ]:
                real_time_plotter, 

In [ ]:
                EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True), 

In [ ]:
                ReduceLROnPlateau(monitor='val_loss', factor=0.001, patience=10)

In [ ]:
            ],

In [ ]:
            verbose=0

In [ ]:
        )      

In [ ]:
    else:

In [ ]:
        model = build_model(numerical_columns, categorical_columns, df, output_size, model_type=model_type, **config)

In [ ]:
        if model_type == 'xgboost':

In [ ]:
            try:

In [ ]:
                preprocessor = model.named_steps['preprocessor']

In [ ]:
                xgb_model = model.named_steps['model']

In [ ]:
                # Fit the preprocessor first

In [ ]:
                preprocessor.fit(combo_train[numerical_columns + categorical_columns])

In [ ]:
                combo_train_transformed = preprocessor.transform(combo_train[numerical_columns + categorical_columns])

In [ ]:
                combo_val_transformed = preprocessor.transform(combo_val[numerical_columns + categorical_columns])

In [ ]:
                # Fit the XGBoost model with early stopping

In [ ]:
                xgb_model.fit(combo_train_transformed, combo_train[y_headers].values,

In [ ]:
                              eval_set=[(combo_val_transformed, combo_val[y_headers].values)],

In [ ]:
                              early_stopping_rounds=10, verbose=False)

In [ ]:
            except Exception as e:

In [ ]:
                print(f"Error during XGBoost training: {e}")

In [ ]:
        else:

In [ ]:
            model.fit(combo_train[numerical_columns + categorical_columns], combo_train[y_headers].values)

In [ ]:
        try:

In [ ]:
            predictions = model.predict(combo_val[numerical_columns + categorical_columns])

In [ ]:
            plot_ml_performance(combo_val[y_headers].values, predictions, f'{model_type.upper()} - {config}')

In [ ]:
        except NotFittedError as e:

In [ ]:
            print(f"Model not fitted error: {e}")

In [ ]:
    end_time = time.time()

In [ ]:
    duration = end_time - start_time

In [ ]:
    task_key = f"{model_type} - {config_str}"

In [ ]:
    task_times[task_key] = duration

In [ ]:
    print(f"Training completed for {task_key} in {duration:.2f} seconds")

In [ ]:
    return model

In [ ]:
def parallel_training(combo, combo_train, combo_val, combo_test, numerical_columns, categorical_columns, y_headers, output_size, df, task_times, mode='parallel'):

In [ ]:
    models = {}

In [ ]:
    basin = combo['BasinTC']

In [ ]:
    formation = combo['FORMATION_CONDENSED']

In [ ]:
    combo_description = f'BasinTC: {basin}, FORMATION_CONDENSED: {formation}'

In [ ]:
    for config in tqdm(configurations, desc=f'Training Neural Networks for {combo_description}'):

In [ ]:
        print(f"Starting training for combination: {combo_description} with config: {config}")

In [ ]:
        model = train_and_evaluate_model(combo_train, combo_val, combo_test, numerical_columns, categorical_columns, y_headers, output_size, config, 'neural_network', df, task_times)

In [ ]:
        models[(basin, formation, str(config))] = model

In [ ]:
        print(f"Training completed for combination: {combo_description} with config: {config}")

In [ ]:
    for ml_config in tqdm(ml_configurations, desc=f'Training ML models for {combo_description}'):

In [ ]:
        print(f"Starting training for combination: {combo_description} with ML config: {ml_config}")

In [ ]:
        model = train_and_evaluate_model(combo_train, combo_val, combo_test, numerical_columns, categorical_columns, y_headers, output_size, {}, ml_config['model_type'], df, task_times)

In [ ]:
        models[(basin, formation, ml_config['model_type'])] = model

In [ ]:
        print(f"Training completed for combination: {combo_description} with ML config: {ml_config}")

In [ ]:
    return models

In [ ]:
def execute_training(specific_combinations, train_df, val_df, test_df, numerical_columns, categorical_columns, y_headers, output_size, df, task_times, mode='parallel'):

In [ ]:
    all_models = {}

In [ ]:
    if mode == 'parallel':

In [ ]:
        with joblib.parallel_backend('loky'):

In [ ]:
            parallel_models = Parallel(n_jobs=-1)(

In [ ]:
                delayed(parallel_training)(

In [ ]:
                    combo[1],  # Access the Series part of the tuple

In [ ]:
                    filter_by_basin_and_formation(train_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                    filter_by_basin_and_formation(val_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                    filter_by_basin_and_formation(test_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                    numerical_columns, categorical_columns, y_headers, output_size, df, task_times, mode

In [ ]:
                ) for combo in tqdm(specific_combinations.iterrows(), total=specific_combinations.shape[0])

In [ ]:
            )

In [ ]:
        for model_dict in parallel_models:

In [ ]:
            all_models.update(model_dict)

In [ ]:
    else:

In [ ]:
        for combo in tqdm(specific_combinations.iterrows(), total=specific_combinations.shape[0]):

In [ ]:
            model_dict = parallel_training(

In [ ]:
                combo[1],  # Access the Series part of the tuple

In [ ]:
                filter_by_basin_and_formation(train_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                filter_by_basin_and_formation(val_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                filter_by_basin_and_formation(test_df, combo[1]['BasinTC'], combo[1]['FORMATION_CONDENSED']),

In [ ]:
                numerical_columns, categorical_columns, y_headers, output_size, df, task_times, mode

In [ ]:
            )

In [ ]:
            all_models.update(model_dict)

In [ ]:
    return all_models

In [ ]:
def compute_shap_values_nn(model, numerical_data, categorical_data):

In [ ]:
    try:

In [ ]:
        combined_data = [numerical_data] + categorical_data

In [ ]:
        explainer = shap.DeepExplainer(model, combined_data)

In [ ]:
        shap_values = explainer.shap_values(combined_data)

In [ ]:
    except Exception as e:

In [ ]:
        logging.error(f"Error using DeepExplainer: {e}. Falling back to KernelExplainer.")

In [ ]:
        combined_data_array = np.concatenate([numerical_data] + categorical_data, axis=1)

In [ ]:
        explainer = shap.KernelExplainer(lambda x: model.predict([x[:, :numerical_data.shape[1]]] + 

In [ ]:
                                                                  [x[:, i].reshape(-1, 1) for i in range(numerical_data.shape[1], combined_data_array.shape[1])]), 

In [ ]:
                                         combined_data_array)

In [ ]:
        shap_values = explainer.shap_values(combined_data_array, nsamples=100)

In [ ]:
    return shap_values

In [ ]:
# Function to compute SHAP values for ML models

In [ ]:
def compute_shap_values_ml(model, numerical_data, categorical_data):

In [ ]:
    combined_data = np.concatenate([numerical_data] + categorical_data, axis=1)

In [ ]:
    try:

In [ ]:
        if hasattr(model, 'named_steps'):

In [ ]:
            model = model.named_steps['model']

In [ ]:
        explainer = shap.TreeExplainer(model)

In [ ]:
        shap_values = explainer.shap_values(combined_data)

In [ ]:
    except Exception as e:

In [ ]:
        logging.error(f"Error using TreeExplainer: {e}. Falling back to KernelExplainer.")

In [ ]:
        explainer = shap.KernelExplainer(model.predict, combined_data)

In [ ]:
        shap_values = explainer.shap_values(combined_data, nsamples=100)

In [ ]:
    return shap_values

In [ ]:
# Function to compute SHAP values for XGBoost

In [ ]:
def compute_shap_values_xgb(model, numerical_data, categorical_data):

In [ ]:
    try:

In [ ]:
        combined_data = np.concatenate([numerical_data] + categorical_data, axis=1)

In [ ]:
        preprocessor = model.named_steps['preprocessor']

In [ ]:
        xgb_model = model.named_steps['model']

In [ ]:
        logging.info("Transforming data using the preprocessor.")

In [ ]:
        combined_data_transformed = preprocessor.transform(pd.DataFrame(combined_data))

In [ ]:
        logging.info("Creating TreeExplainer.")

In [ ]:
        explainer = shap.TreeExplainer(xgb_model)

In [ ]:
        logging.info("Computing SHAP values.")

In [ ]:
        shap_values = explainer.shap_values(combined_data_transformed)

In [ ]:
    except Exception as e:

In [ ]:
        logging.error(f"Error using TreeExplainer: {e}. Falling back to KernelExplainer.")

In [ ]:
        try:

In [ ]:
            explainer = shap.KernelExplainer(xgb_model.predict, combined_data_transformed)

In [ ]:
            shap_values = explainer.shap_values(combined_data_transformed, nsamples=100)

In [ ]:
        except Exception as e2:

In [ ]:
            logging.error(f"Error using KernelExplainer: {e2}")

In [ ]:
            shap_values = None

In [ ]:
    return shap_values

In [ ]:
def compute_shap_values(model, numerical_data, categorical_data, model_type):

In [ ]:
    if model_type == 'neural_network':

In [ ]:
        return compute_shap_values_nn(model, numerical_data, categorical_data)

In [ ]:
    # if model_type == 'xgboost':

In [ ]:
    # elif model_type == 'xgboost':

In [ ]:
    #     return compute_shap_values_xgb(model, numerical_data, categorical_data)

In [ ]:
    # else:

In [ ]:
    #     return compute_shap_values_ml(model, numerical_data, categorical_data)

In [ ]:
    # return none

In [ ]:
def compute_and_log_shap_values(models, test_df, numerical_columns, categorical_columns, shap_sample_size=100):

In [ ]:
    shap_values_dict = {}

In [ ]:
    for (basin, formation, config_str), model in models.items():

In [ ]:
        logging.info(f"Processing SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
        try:

In [ ]:
            combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
            logging.info(f"Filtered test data for {basin} - {formation}")

In [ ]:
            actual_sample_size = min(len(combo_test), shap_sample_size)

In [ ]:
            combo_test_subset = combo_test.sample(n=actual_sample_size, random_state=42)

In [ ]:
            numerical_data = combo_test_subset[numerical_columns].values

In [ ]:
            categorical_data = [combo_test_subset[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
            if 'embedding_output_dim' in config_str:

In [ ]:
                model_type = 'neural_network'

In [ ]:
            # else:

In [ ]:
            #     model_type = 'xgboost' if 'xgboost' in config_str.lower() else 'ml_model'

In [ ]:
            # #model_type = 'xgboost' if 'xgboost' in config_str.lower() else 'ml_model'

In [ ]:
                logging.info(f"Computing SHAP values using {model_type}")

In [ ]:
                shap_values = compute_shap_values(model, numerical_data, categorical_data, model_type)

In [ ]:
                if shap_values is not None:

In [ ]:
                    shap_values_dict[(basin, formation, config_str)] = shap_values

In [ ]:
                    logging.info(f"Successfully computed SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
                else:

In [ ]:
                    logging.error(f"Failed to compute SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
            del combo_test, combo_test_subset, numerical_data, categorical_data, shap_values

In [ ]:
            gc.collect()

In [ ]:
        except Exception as e:

In [ ]:
            logging.error(f"Error computing SHAP values for {basin} - {formation} - {config_str}: {e}")

In [ ]:
    return shap_values_dict

In [ ]:
# Function to compute and log SHAP values

In [ ]:
# def compute_and_log_shap_values(models, test_df, numerical_columns, categorical_columns, shap_sample_size=100):

In [ ]:
#     shap_values_dict = {}

In [ ]:
#     for (basin, formation, config_str), model in models.items():

In [ ]:
#         logging.info(f"Processing SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
#         try:

In [ ]:
#             combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
#             logging.info(f"Filtered test data for {basin} - {formation}")

In [ ]:
#             actual_sample_size = min(len(combo_test), shap_sample_size)

In [ ]:
#             combo_test_subset = combo_test.sample(n=actual_sample_size, random_state=42)

In [ ]:
#             numerical_data = combo_test_subset[numerical_columns].values

In [ ]:
#             categorical_data = [combo_test_subset[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
#             if 'embedding_output_dim' in config_str:

In [ ]:
#                 model_type = 'neural_network'

In [ ]:
#             else:

In [ ]:
#                 model_type = 'xgboost' if 'xgboost' in config_str.lower() else 'ml_model'

In [ ]:
#             logging.info(f"Computing SHAP values using {model_type}")

In [ ]:
#             shap_values = compute_shap_values(model, numerical_data, categorical_data, model_type)

In [ ]:
#             if shap_values is not None:

In [ ]:
#                 shap_values_dict[(basin, formation, config_str)] = shap_values

In [ ]:
#                 logging.info(f"Successfully computed SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
#             else:

In [ ]:
#                 logging.error(f"Failed to compute SHAP values for {basin} - {formation} - {config_str}")

In [ ]:
#             del combo_test, combo_test_subset, numerical_data, categorical_data, shap_values

In [ ]:
#             gc.collect()

In [ ]:
#         except Exception as e:

In [ ]:
#             logging.error(f"Error computing SHAP values for {basin} - {formation} - {config_str}: {e}")

In [ ]:
#     return shap_values_dict

In [ ]:
# def plot_shap_values(shap_values_dict, y_headers, feature_names, output_pdf):

In [ ]:
#     (basin, formation, config_str), shap_values = list(shap_values_dict.items())[0]

In [ ]:
#     # Convert configuration to a more readable string

In [ ]:
#     config_readable = "Neural Network"

In [ ]:
#     # Create a temporary directory to save individual SHAP plots

In [ ]:
#     with tempfile.TemporaryDirectory() as temp_dir:

In [ ]:
#         plot_paths = []

In [ ]:
#         # Generate individual SHAP summary plots

In [ ]:
#         for i in range(shap_values.shape[2]):

In [ ]:
#             fig, ax = plt.subplots(figsize=(8, 12))  # Increase figure size

In [ ]:
#             shap.summary_plot(shap_values[:, :, i], feature_names=feature_names, show=False, plot_type='bar', max_display=len(feature_names))

In [ ]:
#             plt.title(y_headers[i], fontsize=10, fontname='Arial')

In [ ]:
#             ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1e}'))

In [ ]:
#             plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=9, fontname='Arial')

In [ ]:
#             plt.xticks(fontsize=10, fontname='Arial')

In [ ]:
#             plot_path = os.path.join(temp_dir, f"shap_plot_{i}.png")

In [ ]:
#             plt.savefig(plot_path, bbox_inches='tight', dpi=300)  # Save with higher DPI for better quality

In [ ]:
#             plt.close(fig)

In [ ]:
#             plot_paths.append(plot_path)

In [ ]:
#         # Combine individual plots into a single PDF

In [ ]:
#         with PdfPages(output_pdf) as pdf:

In [ ]:
#             fig, axes = plt.subplots(nrows=1, ncols=len(plot_paths), figsize=(len(plot_paths) * 8, 12), sharey=True)

In [ ]:
#             for i, plot_path in enumerate(plot_paths):

In [ ]:
#                 img = plt.imread(plot_path)

In [ ]:
#                 axes[i].imshow(img)

In [ ]:
#                 axes[i].axis('off')

In [ ]:
#                 axes[i].set_title(y_headers[i], fontsize=10, fontname='Arial')

In [ ]:
#             fig.suptitle(config_readable, fontsize=16, fontname='Arial')

In [ ]:
#             plt.subplots_adjust(wspace=0.09)  # Increase spacing between plots

In [ ]:
#             pdf.savefig(fig, dpi=300)  # Save the final PDF with higher DPI

In [ ]:
#             plt.close(fig)

In [ ]:
# # Function to plot SHAP values and save to PDF

In [ ]:
# # def plot_shap_values(shap_values_dict, y_headers, feature_names, output_pdf):

In [ ]:
# #     # Create a temporary directory to save individual SHAP plots

In [ ]:
# #     with tempfile.TemporaryDirectory() as temp_dir:

In [ ]:
# #         plot_paths_dict = {key: [] for key in y_headers}

In [ ]:
# #         # Generate individual SHAP summary plots

In [ ]:
# #         for (basin, formation, config_str), shap_values in shap_values_dict.items():

In [ ]:
# #             config_readable = config_str  # Use config_str for readability

In [ ]:
# #             for i in range(shap_values.shape[2]):

In [ ]:
# #                 fig, ax = plt.subplots(figsize=(8, 12))  # Increase figure size

In [ ]:
# #                 shap.summary_plot(shap_values[:, :, i], feature_names=feature_names, show=False, plot_type='bar', max_display=len(feature_names))

In [ ]:
# #                 plt.title(f"{config_readable} - {y_headers[i]}", fontsize=10, fontname='Arial')

In [ ]:
# #                 ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1e}'))

In [ ]:
# #                 plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=9, fontname='Arial')

In [ ]:
# #                 plt.xticks(fontsize=10, fontname='Arial')

In [ ]:
# #                 plot_path = os.path.join(temp_dir, f"shap_plot_{config_str}_{i}.png")

In [ ]:
# #                 plt.savefig(plot_path, bbox_inches='tight', dpi=300)  # Save with higher DPI for better quality

In [ ]:
# #                 plt.close(fig)

In [ ]:
# #                 plot_paths_dict[y_headers[i]].append(plot_path)

In [ ]:
# #         # Combine individual plots into a single PDF

In [ ]:
# #         with PdfPages(output_pdf) as pdf:

In [ ]:
# #             for y_header in y_headers:

In [ ]:
# #                 fig, axes = plt.subplots(nrows=len(plot_paths_dict[y_header]), ncols=1, figsize=(8, len(plot_paths_dict[y_header]) * 12), sharey=True)

In [ ]:
# #                 for j, plot_path in enumerate(plot_paths_dict[y_header]):

In [ ]:
# #                     img = plt.imread(plot_path)

In [ ]:
# #                     axes[j].imshow(img)

In [ ]:
# #                     axes[j].axis('off')

In [ ]:
# #                 fig.suptitle(f"SHAP for {y_header}", fontsize=16, fontname='Arial')

In [ ]:
# #                 plt.subplots_adjust(hspace=0.3)  # Adjust spacing between plots

In [ ]:
# #                 pdf.savefig(fig, dpi=300)  # Save the final PDF with higher DPI

In [ ]:
# #                 plt.close(fig)

In [ ]:
# Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Assuming df, train_df, val_df, test_df are defined and contain the necessary columns

In [ ]:
# unique_combinations = df[['BasinTC', 'FORMATION_CONDENSED']].drop_duplicates()

In [ ]:
configurations = [

In [ ]:
    {'embedding_output_dim': 20, 'dense_layer_sizes': [256, 128, 64, 32], 'dropout_rate': 0.3, 'regularization': l2(0.01), 'activation': 'relu', 'optimizer': 'adam', 'loss_function': 'mse'},

In [ ]:
 #   {'embedding_output_dim': 15, 'dense_layer_sizes': [64, 32], 'dropout_rate': 0.2, 'regularization': l1(0.01), 'activation': 'relu', 'optimizer': 'rmsprop', 'loss_function': 'mse'}

In [ ]:
]

In [ ]:
# ml_configurations = [

In [ ]:
# ]

In [ ]:
ml_configurations = [

In [ ]:
    {'model_type': 'random_forest'},

In [ ]:
    {'model_type': 'decision_tree'},

In [ ]:
    {'model_type': 'xgboost'}

In [ ]:
]

In [ ]:
# Assuming df, train_df, val_df, test_df are defined and contain the necessary columns

In [ ]:
specific_combinations = df[

In [ ]:
    (df['BasinTC'] == 'Midland') & 

In [ ]:
    (df['FORMATION_CONDENSED'].isin(['LSS', 'WCA', 'WCB', 'WCD']))

In [ ]:
    #(df['FORMATION_CONDENSED'].isin(['WCA']))

In [ ]:
].drop_duplicates(subset=['BasinTC', 'FORMATION_CONDENSED'])

In [ ]:
output_size = len(train_df[y_headers].columns)  # Ensure output_size is defined

In [ ]:
# Dictionary to store task times

In [ ]:
task_times = {}

In [ ]:
# Execution

In [ ]:
models = execute_training(

In [ ]:
    specific_combinations, train_df, val_df, test_df, numerical_columns, categorical_columns, y_headers, output_size, df, task_times, mode='serial'  # Change mode to 'serial' for serial execution

In [ ]:
)

In [ ]:
# Compute SHAP values

In [ ]:
#shap_values_dict = compute_and_log_shap_values(models, test_df, numerical_columns, categorical_columns, shap_sample_size=100)

In [ ]:
# Print out the task times to identify slow tasks

In [ ]:
print("Task times (in seconds):")

In [ ]:
for task, duration in task_times.items():

In [ ]:
    print(f"{task}: {duration:.2f} seconds")

In [ ]:
# # Define the variables

In [ ]:
# basin = 'Midland'

In [ ]:
# #formation = 'WCA'

In [ ]:
# formation = ['LSS', 'WCA', 'WCB', 'WCD']

In [ ]:
# config_str = "{'embedding_output_dim': 20, 'dense_layer_sizes': [256, 128, 64, 32], 'dropout_rate': 0.3, 'regularization': <keras.src.regularizers.regularizers.L2 object at 0x0000024A7D101640>, 'activation': 'relu', 'optimizer': 'adam', 'loss_function': 'mse'}"

In [ ]:
# feature_names = numerical_columns + categorical_columns  # Ensure this list contains all your feature names

In [ ]:
# output_pdf = r"C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\shap_summary_plot.pdf"

In [ ]:
# # Plot the SHAP values

In [ ]:
# plot_shap_values(shap_values_dict, y_headers, feature_names, output_pdf)

In [ ]:
# Function to denormalize numerical data

In [ ]:
def denormalize_data_input(data, scaler):

In [ ]:
    return scaler.inverse_transform(data)

In [ ]:
# Function to denormalize data and revert log transformation

In [ ]:
def denormalize_data_output(data, scaler, log_transform_columns):

In [ ]:
    denormalized_data = scaler.inverse_transform(data)

In [ ]:
    denormalized_df = pd.DataFrame(denormalized_data, columns=scaler.get_feature_names_out())

In [ ]:
    denormalized_df[log_transform_columns] = np.expm1(denormalized_df[log_transform_columns])

In [ ]:
    if any('BuildupRate' in col for col in log_transform_columns):

In [ ]:
        denormalized_df[log_transform_columns] = denormalized_df[log_transform_columns].replace(1, 0)

In [ ]:
    return denormalized_df

In [ ]:
# Function to decode categorical data

In [ ]:
def decode_categorical(data, encoders, column_names):

In [ ]:
    decoded_data = {}

In [ ]:
    for i, col in enumerate(column_names):

In [ ]:
        le = encoders[col]

In [ ]:
        decoded_data[col] = le.inverse_transform(data[:, i].astype(int))

In [ ]:
    return decoded_data

In [ ]:
# Denormalize and decode the training DataFrame

In [ ]:
def denormalize_and_decode(train_df, numerical_columns, categorical_columns, y_headers, input_scaler, output_scaler, encoders):

In [ ]:
    # Denormalize numerical features

In [ ]:
    train_df[numerical_columns] = denormalize_data_input(train_df[numerical_columns], input_scaler)

In [ ]:
    #log_transform_columns = [col for col in y_headers if 'BuildupRate' in col or 'InitialProd' in col]

In [ ]:
    log_transform_columns = [col for col in y_headers if 'InitialProd' in col]

In [ ]:
    # Denormalize target features

In [ ]:
    train_df[y_headers] = denormalize_data_output(train_df[y_headers], output_scaler, log_transform_columns)

In [ ]:
    # Decode categorical features

In [ ]:
    decoded_categorical = decode_categorical(train_df, encoders, categorical_columns)

In [ ]:
    for col in categorical_columns:

In [ ]:
        train_df[col] = decoded_categorical[col]

In [ ]:
    return train_df

In [ ]:
# Function to generate hyperbolic decline curve

In [ ]:
def hyperbolic_decline(t, qi, di, b):

In [ ]:
    return qi / ((1 + b * di * t) ** (1/b))

In [ ]:
def modified_hyperbolic(time, qi, di, b, Dlim, IBU, MBU, buildup_method='Linear', epsilon=1e-10):

In [ ]:
    """

In [ ]:
    Calculate the production rate using the Modified Hyperbolic Decline Model with a buildup phase.

In [ ]:
    :param time: array-like, time in months

In [ ]:
    :param qi: float, initial production rate

In [ ]:
    :param di: float, initial decline rate

In [ ]:
    :param b: float, hyperbolic coefficient

In [ ]:
    :param Dlim: float, limited decline rate

In [ ]:
    :param IBU: float, initial buildup rate

In [ ]:
    :param MBU: float, months in production for buildup

In [ ]:
    :param buildup_method: str, method for buildup ('Flat', 'Linear', 'Exp')

In [ ]:
    :return: array-like, production rates

In [ ]:
    """

In [ ]:
    # Convert decline rates to nominal rates

In [ ]:
    di = ((1 - 0.01 * di) ** (-b) - 1) / (12 * b)

In [ ]:
    Dlim = ((1 - 0.01 * Dlim) ** (-b) - 1) / (12 * b)

In [ ]:
    # Calculate switch point

In [ ]:
    qlim = qi * (Dlim / di) ** (1.0 / b)

In [ ]:
    tlim = ((qi / qlim) ** (b) - 1.0) / (b * di)

In [ ]:
    # Separate time into buildup, hyperbolic, and exponential segments

In [ ]:
    t_buildup = time[time <= MBU]

In [ ]:
    t_post_buildup = time[time > MBU]

In [ ]:
    t_hyp = t_post_buildup[t_post_buildup < tlim]

In [ ]:
    t_exp = t_post_buildup[t_post_buildup >= tlim]

In [ ]:
    # Calculate production during buildup period

In [ ]:
    if buildup_method == 'Flat':

In [ ]:
        buildup_production = IBU + np.zeros_like(t_buildup)

In [ ]:
    elif buildup_method == 'Linear':

In [ ]:
        slope = (qi - IBU) / (MBU + epsilon)

In [ ]:
        buildup_production = IBU + slope * t_buildup

In [ ]:
    elif buildup_method == 'Exp':

In [ ]:
        slope = np.log(qi / (IBU + epsilon)) / (MBU + epsilon)

In [ ]:
        buildup_production = IBU * np.exp(slope * t_buildup)

In [ ]:
    else:

In [ ]:
        raise ValueError("Unknown buildup method: {}".format(buildup_method))

In [ ]:
    # Calculate production during hyperbolic decline period

In [ ]:
    q_model_hyp = qi * (1.0 + b * di * (t_hyp - MBU)) ** (-1.0 / b)

In [ ]:
    # Calculate production during exponential decline period

In [ ]:
    q_model_exp = qlim * np.exp(-Dlim * (t_exp - tlim))

In [ ]:
    # Combine production rates

In [ ]:
    production_post_buildup = np.concatenate((q_model_hyp, q_model_exp))

In [ ]:
    production = np.concatenate((buildup_production, production_post_buildup))

In [ ]:
    return production

In [ ]:
# Function to plot decline curves in semi-logarithmic scale

In [ ]:
def plot_decline_curves(time, actual, predicted, title):

In [ ]:
    plt.figure(figsize=(10, 6))

In [ ]:
    plt.plot(time, actual, 'b-o', label='Actual')

In [ ]:
    plt.plot(time, predicted, 'r--x', label='Predicted')

In [ ]:
    plt.yscale('log')

In [ ]:
    plt.ylim(1, 100000)

In [ ]:
    plt.title(title)

In [ ]:
    plt.xlabel('Time (Months)')

In [ ]:
    plt.ylabel('Production Rate (bbl/day)')

In [ ]:
    plt.legend()

In [ ]:
    plt.grid(True, which='both', linestyle='--')

In [ ]:
    plt.show()

In [ ]:
# Function to generate production rates

In [ ]:
def generate_production_rates(y_pred_denormalized, y_true_denormalized, headers, time):

In [ ]:
    predicted_productions = []

In [ ]:
    actual_productions = []

In [ ]:
    for idx in range(len(y_pred_denormalized)):

In [ ]:
        qi_pred = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_InitialProd')]

In [ ]:
        di_pred = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_DiCoefficient')]

In [ ]:
        b_pred  = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_BCoefficient')]

In [ ]:
        IBU_pred = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_BuildupRate')]

In [ ]:
        MBU_pred = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_MonthsInProd')]

In [ ]:
        Dlim_pred = y_pred_denormalized.iloc[idx, headers.index('Oil_Params_P50_LimDeclineRate')]

In [ ]:
        qi_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_InitialProd')]

In [ ]:
        di_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_DiCoefficient')]

In [ ]:
        b_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_BCoefficient')]

In [ ]:
        IBU_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_BuildupRate')]

In [ ]:
        MBU_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_MonthsInProd')]

In [ ]:
        Dlim_true = y_true_denormalized.iloc[idx, headers.index('Oil_Params_P50_LimDeclineRate')]

In [ ]:
        predicted_production = modified_hyperbolic(time, qi_pred, di_pred, b_pred, Dlim_pred, IBU_pred, MBU_pred)

In [ ]:
        actual_production = modified_hyperbolic(time, qi_true, di_true, b_true, Dlim_true, IBU_true, MBU_true)

In [ ]:
        # predicted_production = hyperbolic_decline(time, qi_pred, di_pred, b_pred)

In [ ]:
        # actual_production = hyperbolic_decline(time, qi_true, di_true, b_true)

In [ ]:
        predicted_productions.append(predicted_production)

In [ ]:
        actual_productions.append(actual_production)

In [ ]:
    return predicted_productions, actual_productions

In [ ]:
# Function to calculate errors

In [ ]:
def calculate_errors(predicted_productions, actual_productions):

In [ ]:
    errors = []

In [ ]:
    for pred, actual in zip(predicted_productions, actual_productions):

In [ ]:
        mse = mean_squared_error(actual, pred)

In [ ]:
        errors.append(mse)

In [ ]:
    return errors

In [ ]:
# Function to identify best and worst matches

In [ ]:
def identify_best_worst_matches(errors, y_true_denormalized, y_pred_denormalized):

In [ ]:
    best_match_idx = np.argmin(errors)

In [ ]:
    worst_match_idx = np.argmax(errors)

In [ ]:
    best_match_true = y_true_denormalized.iloc[best_match_idx]

In [ ]:
    best_match_pred = y_pred_denormalized.iloc[best_match_idx]

In [ ]:
    worst_match_true = y_true_denormalized.iloc[worst_match_idx]

In [ ]:
    worst_match_pred = y_pred_denormalized.iloc[worst_match_idx]

In [ ]:
    return best_match_true, best_match_pred, worst_match_true, worst_match_pred, best_match_idx, worst_match_idx

In [ ]:
# Function to plot decline curves in semi-logarithmic scale

In [ ]:
def plot_decline_curves(time, actual, predicted, title):

In [ ]:
    plt.figure(figsize=(10, 6))

In [ ]:
    plt.plot(time, actual, 'b-o', label='Actual')

In [ ]:
    plt.plot(time, predicted, 'r--x', label='Predicted')

In [ ]:
    plt.yscale('log')

In [ ]:
    plt.ylim(1, 100000)

In [ ]:
    plt.title(title)

In [ ]:
    plt.xlabel('Time (Months)')

In [ ]:
    plt.ylabel('Production Rate (bbl/day)')

In [ ]:
    plt.legend()

In [ ]:
    plt.grid(True, which='both', linestyle='--')

In [ ]:
    plt.show()

In [ ]:
# Define the prediction function

In [ ]:
#@tf.function(reduce_retracing=True)

In [ ]:
def predict_with_model(model, numerical_data, categorical_data):

In [ ]:
    if isinstance(model, Model):  # Keras model

In [ ]:
        return model.predict([numerical_data] + categorical_data)

In [ ]:
    else:  # Sklearn model

In [ ]:
        import pandas as pd

In [ ]:
        data_combined = np.hstack([numerical_data] + categorical_data)

In [ ]:
        combined_df = pd.DataFrame(data_combined, columns=numerical_columns + categorical_columns)

In [ ]:
        return model.predict(combined_df)

In [ ]:
def combine_data(numerical_data, categorical_data):

In [ ]:
    if categorical_data:  # Check if categorical_data is not empty

In [ ]:
        categorical_data_combined = np.hstack([cat for cat in categorical_data])

In [ ]:
        combined_data = np.hstack((numerical_data, categorical_data_combined))

In [ ]:
    else:

In [ ]:
        combined_data = numerical_data

In [ ]:
    return combined_data

In [ ]:
def sensitivity_analysis(cov_matrix, model, numerical_columns, categorical_columns, combo_val, time, y_headers, y_true_denormalized, y_pred_denormalized, output_scaler, log_transform_columns):

In [ ]:
    combined_val_data = combine_data(combo_val[numerical_columns].values, 

In [ ]:
                                     [combo_val[col].astype(int).values.reshape(-1, 1) for col in categorical_columns])

In [ ]:
    sensitivities = []

In [ ]:
    feature_names = numerical_columns + categorical_columns

In [ ]:
    # Debugging: Print the shape of combined_val_data

In [ ]:
    print(f"Shape of combined_val_data: {combined_val_data.shape}")

In [ ]:
    for i, feature in enumerate(feature_names):

In [ ]:
        perturbed_data = combined_val_data.copy()

In [ ]:
        perturbed_data[:, i] += np.sqrt(cov_matrix[i, i])  # Perturb by one standard deviation

In [ ]:
        # Predict using the model

In [ ]:
        perturbed_numerical_data = perturbed_data[:, :len(numerical_columns)]

In [ ]:
        perturbed_categorical_data = [perturbed_data[:, len(numerical_columns) + j].reshape(-1, 1) for j in range(len(categorical_columns))]

In [ ]:
        y_pred_perturbed = predict_with_model(model, perturbed_numerical_data, perturbed_categorical_data)

In [ ]:
        # Denormalize the predictions

In [ ]:
        y_pred_perturbed_denormalized = denormalize_data_output(y_pred_perturbed, output_scaler, log_transform_columns)

In [ ]:
        # Generate production rates

In [ ]:
        predicted_productions_perturbed, _ = generate_production_rates(y_pred_perturbed_denormalized, y_true_denormalized, y_headers, time)

In [ ]:
        # Calculate mean sensitivity (change in production rate)

In [ ]:
        mean_sensitivity = np.mean(np.abs(np.array(predicted_productions_perturbed) - np.array(predicted_productions)), axis=0)

In [ ]:
        sensitivities.append(mean_sensitivity)

In [ ]:
    return sensitivities, feature_names

In [ ]:
def plot_sensitivity_analysis(sensitivities, feature_names):

In [ ]:
    plt.figure(figsize=(12, 6))

In [ ]:
    plt.barh(feature_names, sensitivities)

In [ ]:
    plt.xlabel('Mean Sensitivity')

In [ ]:
    plt.ylabel('Input Features')

In [ ]:
    plt.title('Sensitivity Analysis of Predicted Production Rates vs Input Features')

In [ ]:
    plt.grid(True)

In [ ]:
    plt.show()

In [ ]:
years = 5

In [ ]:
time_array = np.linspace(1, 12 * years, 12 * years)  # 5 Years

In [ ]:
# Evaluate the models

In [ ]:
evaluation_results = {}

In [ ]:
best_performing_models = {}

In [ ]:
for (basin, formation, config_str), model in models.items():

In [ ]:
    # Filter the test data for the specific basin and formation

In [ ]:
    combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
    numerical_data = combo_test[numerical_columns].values

In [ ]:
    categorical_data = [combo_test[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
    # Prepare the target values (y_true)

In [ ]:
    y_true = combo_test[y_headers].values  # Use entire y_headers as output

In [ ]:
    # Predict outputs using the model

In [ ]:
    y_pred = predict_with_model(model, numerical_data, categorical_data)

In [ ]:
    # Denormalize predictions and actual values

In [ ]:
    y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
    y_true_denormalized = denormalize_data_output(y_true, output_scaler, log_transform_columns)

In [ ]:
    # Generate production rates

In [ ]:
    predicted_productions, actual_productions = generate_production_rates(y_pred_denormalized, y_true_denormalized, y_headers, time_array)

In [ ]:
    # Calculate errors

In [ ]:
    errors = calculate_errors(predicted_productions, actual_productions)

In [ ]:
    # Identify best and worst matches

In [ ]:
    best_match_true, best_match_pred, worst_match_true, worst_match_pred, best_match_idx, worst_match_idx = identify_best_worst_matches(errors, y_true_denormalized, y_pred_denormalized)

In [ ]:
    # Plot the best match

In [ ]:
    best_actual_production = actual_productions[best_match_idx]

In [ ]:
    best_predicted_production = predicted_productions[best_match_idx]

In [ ]:
    plot_decline_curves(time_array, best_actual_production, best_predicted_production, f'Best Match Hyperbolic Decline Curve for {basin} - {formation}')

In [ ]:
    # Plot the worst match

In [ ]:
    worst_actual_production = actual_productions[worst_match_idx]

In [ ]:
    worst_predicted_production = predicted_productions[worst_match_idx]

In [ ]:
    plot_decline_curves(time_array, worst_actual_production, worst_predicted_production, f'Worst Match Hyperbolic Decline Curve for {basin} - {formation}')

In [ ]:
    # Calculating evaluation metrics

In [ ]:
    if y_true_denormalized.shape == y_pred_denormalized.shape:

In [ ]:
        mse = mean_squared_error(y_true_denormalized, y_pred_denormalized)

In [ ]:
        mae = mean_absolute_error(y_true_denormalized, y_pred_denormalized)

In [ ]:
        evaluation_results[(basin, formation, config_str)] = {'MSE': mse, 'MAE': mae}

In [ ]:
        print(f"Performance for Basin: {basin}, Formation: {formation}, Config: {config_str} - MSE: {mse:.4f}, MAE: {mae:.4f}")

In [ ]:
        # Update best performing model

In [ ]:
        if (basin, formation) not in best_performing_models or mse < best_performing_models[(basin, formation)]['MSE']:

In [ ]:
            best_performing_models[(basin, formation)] = {'MSE': mse, 'MAE': mae, 'config_str': config_str, 'model': model}

In [ ]:
    else:

In [ ]:
        print(f"Shape mismatch between y_true {y_true_denormalized.shape} and y_pred {y_pred_denormalized.shape}, check data preparation steps.")

In [ ]:
# Print the best performing model for each basin and formation

In [ ]:
for (basin, formation), best_model_info in best_performing_models.items():

In [ ]:
    print(f"Best Performing Model for Basin: {basin}, Formation: {formation} - Config: {best_model_info['config_str']} - MSE: {best_model_info['MSE']:.4f}, MAE: {best_model_info['MAE']:.4f}")

In [ ]:
# Directory to save individual plots

In [ ]:
output_dir = 'C:/Users/Prakhar.Sarkar/OneDrive - SRP Management Services/Documents/_For_Prakhar/covarianceplots/plots'

In [ ]:
os.makedirs(output_dir, exist_ok=True)

In [ ]:
#plots_by_combination = {}

In [ ]:
plots_by_combination = {}

In [ ]:
for (basin, formation, config_str), model in models.items():

In [ ]:
    # Check if config_str indicates a neural network configuration

In [ ]:
    if 'embedding_output_dim' in config_str or 'dense_layer_sizes' in config_str:

In [ ]:
        model_type = 'Neural Network'

In [ ]:
    else:

In [ ]:
        model_type = config_str.split(' ')[0]  # Use the first part of the config_str as the model type

In [ ]:
    combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
    numerical_data = combo_test[numerical_columns].values

In [ ]:
    categorical_data = [combo_test[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
    y_true = combo_test[y_headers].values

In [ ]:
    y_pred = predict_with_model(model, numerical_data, categorical_data)

In [ ]:
    y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
    y_true_denormalized = denormalize_data_output(y_true, output_scaler, log_transform_columns)

In [ ]:
    years = 5

In [ ]:
    time_array = np.linspace(1, 12 * years, 12 * years)  # 5 Years

In [ ]:
    # Generate production rates for each sample

In [ ]:
    predicted_productions_list = []

In [ ]:
    for i in range(y_pred_denormalized.shape[0]):

In [ ]:
        qi_pred = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_InitialProd')]

In [ ]:
        di_pred = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_DiCoefficient')]

In [ ]:
        b_pred  = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_BCoefficient')]

In [ ]:
        IBU_pred = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_BuildupRate')]

In [ ]:
        MBU_pred = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_MonthsInProd')]

In [ ]:
        Dlim_pred = y_pred_denormalized.iloc[i, y_headers.index('Oil_Params_P50_LimDeclineRate')]

In [ ]:
        predicted_production = modified_hyperbolic(time_array, qi_pred, di_pred, b_pred, Dlim_pred, IBU_pred, MBU_pred)

In [ ]:
        predicted_productions_list.append(predicted_production)

In [ ]:
    predicted_productions_flat = np.array(predicted_productions_list).flatten()

In [ ]:
    # Check if categorical columns are empty

In [ ]:
    if categorical_columns:

In [ ]:
        categorical_dummies = pd.get_dummies(combo_test[categorical_columns], drop_first=True)

In [ ]:
        all_features = pd.concat([combo_test[numerical_columns], categorical_dummies], axis=1)

In [ ]:
    else:

In [ ]:
        all_features = combo_test[numerical_columns]

In [ ]:
    # Repeat features to match the length of flattened predicted productions

In [ ]:
    repeated_features = np.repeat(all_features.values, len(time_array), axis=0)

In [ ]:
    # Ensure the repeated_features array matches the size of the predicted production rates

In [ ]:
    if len(repeated_features) != len(predicted_productions_flat):

In [ ]:
        raise ValueError(f"Feature DataFrame and predicted productions have different lengths: {len(repeated_features)} vs {len(predicted_productions_flat)}")

In [ ]:
    # Calculate covariance

In [ ]:
    cov_matrix = np.cov(repeated_features.T, predicted_productions_flat)

In [ ]:
    feature_names = all_features.columns

In [ ]:
    sensitivity = cov_matrix[:-1, -1]

In [ ]:
    # Create a separate figure for each combination

In [ ]:
    plt.figure(figsize=(14, 20))

In [ ]:
    sns.barplot(x=sensitivity, y=feature_names)

In [ ]:
    plt.title(f'{basin}-{formation} ({model_type})', fontsize=16, fontweight='bold')

In [ ]:
    plt.xlabel('Covariance', fontsize=12, fontname='Arial')

In [ ]:
    plt.ylabel('Input Features', fontsize=14, fontweight='bold', fontname='Arial')

In [ ]:
    plt.xticks(fontsize=10, fontname='Arial')

In [ ]:
    plt.yticks(fontsize=10, fontname='Arial')

In [ ]:
    plt.tight_layout()

In [ ]:
    # Save the plot

In [ ]:
    plot_filename = os.path.join(output_dir, f'{basin}_{formation}_{model_type}.jpeg')

In [ ]:
    plt.savefig(plot_filename, format='jpeg',dpi=(500))

In [ ]:
    plt.close()

In [ ]:
    # Store plot filename for grid organization

In [ ]:
    if (basin, formation) not in plots_by_combination:

In [ ]:
        plots_by_combination[(basin, formation)] = []

In [ ]:
    plots_by_combination[(basin, formation)].append(plot_filename)

In [ ]:
# Load the saved plots and stitch them together into a grid for each combination

In [ ]:
for (basin, formation), plot_filenames in plots_by_combination.items():

In [ ]:
    images = [Image.open(filename) for filename in plot_filenames]

In [ ]:
    # Determine the number of rows and columns for the grid

In [ ]:
    num_plots = len(images)

In [ ]:
    num_cols = 4  # Set the number of columns for the grid

In [ ]:
    num_rows = (num_plots + num_cols - 1) // num_cols

In [ ]:
    # Get the dimensions of the images

In [ ]:
    image_width, image_height = images[0].size

In [ ]:
    # Create a new blank image for the grid

In [ ]:
    grid_image = Image.new('RGB', (num_cols * image_width, num_rows * image_height))

In [ ]:
    # Paste each plot image into the grid

In [ ]:
    for index, image in enumerate(images):

In [ ]:
        row = index // num_cols

In [ ]:
        col = index % num_cols

In [ ]:
        grid_image.paste(image, (col * image_width, row * image_height))

In [ ]:
    # Save the final grid image for the combination

In [ ]:
    grid_image.save(f'C:/Users/Prakhar.Sarkar/OneDrive - SRP Management Services/Documents/_For_Prakhar/covarianceplots/{basin}_{formation}_grid_image.jpeg', format='jpeg')

In [ ]:
# plots_by_combination = {}

In [ ]:
# for (basin, formation, config_str), model in models.items():

In [ ]:
#     # Check if config_str indicates a neural network configuration

In [ ]:
#     if 'embedding_output_dim' in config_str or 'dense_layer_sizes' in config_str:

In [ ]:
#         model_type = 'Neural Network'

In [ ]:
#     else:

In [ ]:
#         model_type = config_str.split(' ')[0]  # Use the first part of the config_str as the model type

In [ ]:
#     combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
#     numerical_data = combo_test[numerical_columns].values

In [ ]:
#     categorical_data = [combo_test[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
#     # Debugging: Print shapes

In [ ]:
#     print(f"Processing {basin}-{formation} ({model_type})")

In [ ]:
#     print(f"Numerical data shape: {numerical_data.shape}")

In [ ]:
#     if model_type == 'Neural Network':

In [ ]:
#         # Use KernelExplainer for neural networks

In [ ]:
#         explainer = shap.KernelExplainer(model.predict, numerical_data)

In [ ]:
#         shap_values = explainer.shap_values(numerical_data, nsamples=100)

In [ ]:
#     # else:

In [ ]:
#     #     if isinstance(model, Pipeline):

In [ ]:
#     #         model = model.named_steps['model']

In [ ]:
#     #     # Example for tree-based models (RandomForest, XGBoost)

In [ ]:
#     #     # Use TreeExplainer for tree-based models

In [ ]:
#     #     explainer = shap.TreeExplainer(model)

In [ ]:
#     #     shap_values = explainer.shap_values(combo_test[numerical_columns])

In [ ]:
#     # Debugging: Print shapes of SHAP values

In [ ]:
#     print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
#     # Ensure SHAP values have the correct shape

In [ ]:
#     if isinstance(shap_values, list):

In [ ]:
#         shap_values = shap_values[0]  # Select the first output if it's a list of arrays

In [ ]:
#     # If SHAP values have three dimensions, select the appropriate slice

In [ ]:
#     if shap_values.ndim == 3:

In [ ]:
#         shap_values = shap_values[:, :, 0]  # Assuming we are interested in the first output

In [ ]:
#     # Summary plot of SHAP values

In [ ]:
#     shap.summary_plot(shap_values, combo_test[numerical_columns], plot_type="bar")

In [ ]:
#     plt.title(f'{basin}-{formation} ({model_type}) SHAP Summary')

In [ ]:
#     plt.tight_layout()

In [ ]:
#     plt.show()

In [ ]:
def ensure_directory_exists(directory):

In [ ]:
    if not os.path.exists(directory):

In [ ]:
        os.makedirs(directory)

In [ ]:
def find_best_performing_index(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns):

In [ ]:
    best_performing_indices = {}

In [ ]:
    for (basin, formation, config_str), model in models.items():

In [ ]:
        combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
        numerical_data = combo_test[numerical_columns].values

In [ ]:
        categorical_data = [combo_test[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
        y_true = combo_test[y_headers].values

In [ ]:
        y_pred = predict_with_model(model, numerical_data, categorical_data)

In [ ]:
        y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
        y_true_denormalized = denormalize_data_output(y_true, output_scaler, log_transform_columns)

In [ ]:
        mse = mean_squared_error(y_true_denormalized, y_pred_denormalized, multioutput='raw_values')

In [ ]:
        if mse.size > 0:

In [ ]:
            best_match_idx = mse.mean(axis=0).argmin()

In [ ]:
            if (basin, formation) not in best_performing_indices or mse.mean() < best_performing_indices[(basin, formation)]['mse']:

In [ ]:
                best_performing_indices[(basin, formation)] = {

In [ ]:
                    'index': best_match_idx,

In [ ]:
                    'mse': mse.mean()

In [ ]:
                }

In [ ]:
    return best_performing_indices

In [ ]:
def identify_best_performing_rows(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns, best_performing_indices):

In [ ]:
    best_performing_rows = {}

In [ ]:
    for (basin, formation, config_str), model in models.items():

In [ ]:
        combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
        best_match_idx = best_performing_indices[(basin, formation)]['index']

In [ ]:
        best_match_row = combo_test.iloc[best_match_idx]

In [ ]:
        best_performing_rows[(basin, formation, config_str)] = {

In [ ]:
            'model': model,

In [ ]:
            'best_match_row': best_match_row

In [ ]:
        }

In [ ]:
    return best_performing_rows

In [ ]:
def generate_sensitivity_data(best_performing_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array):

In [ ]:
    variations = {

In [ ]:
        'HORIZONTIAL_WELL_LENGTH': np.arange(8000, 11500, 500),

In [ ]:
        'ProppantPerFoot': np.arange(1000, 2200, 200),

In [ ]:
        'FluidPerFoot_bblft': np.arange(40, 70, 10)

In [ ]:
    }

In [ ]:
    results = {key: [] for key in variations.keys()}

In [ ]:
    for param, values in variations.items():

In [ ]:
        for value in values:

In [ ]:
            modified_row = best_performing_row.copy()

In [ ]:
            modified_row[param] = value

In [ ]:
            scaled_numerical = input_scaler.transform(pd.DataFrame([modified_row[numerical_columns]], columns=numerical_columns))

In [ ]:
            categorical_data = [modified_row[col].astype(int).reshape(-1, 1) for col in categorical_columns]

In [ ]:
            y_pred = predict_with_model(model, scaled_numerical, categorical_data)

In [ ]:
            y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
            # Convert to NumPy array for indexing

In [ ]:
            y_pred_denormalized = np.array(y_pred_denormalized)

In [ ]:
            production_rates = modified_hyperbolic(time_array, 

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_InitialProd')],

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_DiCoefficient')],

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_BCoefficient')],

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_LimDeclineRate')],

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_BuildupRate')],

In [ ]:
                                                   y_pred_denormalized[0, y_headers.index('Oil_Params_P50_MonthsInProd')])

In [ ]:
            results[param].append((value, production_rates))

In [ ]:
    return results

In [ ]:
def plot_sensitivity_analysis(results, time_array, param, ax, method):

In [ ]:
    for value, production in results[param]:

In [ ]:
        ax.plot(time_array, production, label=f'{param}={value}')

In [ ]:
    ax.set_yscale('log')

In [ ]:
    ax.set_xlabel('Time (Months)', fontsize=12, fontname='Arial')

In [ ]:
    ax.set_ylabel('Production Rate (bbl/day)', fontsize=12, fontname='Arial')

In [ ]:
    wrapped_title = "\n".join(wrap(f'Sensitivity Analysis for {param} ({method})', 30))

In [ ]:
    ax.set_title(wrapped_title, fontsize=14, fontweight='bold', fontname='Arial')

In [ ]:
    ax.legend()

In [ ]:
    ax.grid(True, which='both', linestyle='--')

In [ ]:
def save_combined_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir):

In [ ]:
    plots_by_combination = {}

In [ ]:
    for (basin, formation, config_str), info in best_performing_rows.items():

In [ ]:
        model = info['model']

In [ ]:
        best_match_row = info['best_match_row']

In [ ]:
        results = generate_sensitivity_data(best_match_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array)

In [ ]:
        method = 'Neural Network' if 'embedding_output_dim' in config_str else config_str.split(' ')[0]

In [ ]:
        if (basin, formation) not in plots_by_combination:

In [ ]:
            plots_by_combination[(basin, formation)] = {'methods': [], 'results': []}

In [ ]:
        plots_by_combination[(basin, formation)]['methods'].append(method)

In [ ]:
        plots_by_combination[(basin, formation)]['results'].append(results)

In [ ]:
    for (basin, formation), data in plots_by_combination.items():

In [ ]:
        methods = data['methods']

In [ ]:
        results_list = data['results']

In [ ]:
        fig, axs = plt.subplots(len(methods), 3, figsize=(20, 6 * len(methods)))

In [ ]:
        fig.subplots_adjust(hspace=0.6, wspace=0.4)

In [ ]:
        for i, (method, results) in enumerate(zip(methods, results_list)):

In [ ]:
            for j, param in enumerate(results.keys()):

In [ ]:
                plot_sensitivity_analysis(results, time_array, param, axs[i, j], method)

In [ ]:
        overall_title = f'Sensitivity Analysis for {basin} - {formation}'

In [ ]:
        fig.suptitle(overall_title, fontsize=16, fontweight='bold', fontname='Arial')

In [ ]:
        basin_formation_dir = os.path.join(output_dir, f"{basin}_{formation}")

In [ ]:
        ensure_directory_exists(basin_formation_dir)

In [ ]:
        plot_filename = os.path.join(basin_formation_dir, f'sensitivity_analysis_combined.png')

In [ ]:
        plt.savefig(plot_filename, format='png', dpi=300)

In [ ]:
        plt.close()

In [ ]:
def save_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir):

In [ ]:
    for (basin, formation, config_str), info in best_performing_rows.items():

In [ ]:
        model = info['model']

In [ ]:
        best_match_row = info['best_match_row']

In [ ]:
        results = generate_sensitivity_data(best_match_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array)

In [ ]:
        method = 'Neural Network' if 'embedding_output_dim' in config_str else config_str.split(' ')[0]

In [ ]:
        fig, axs = plt.subplots(1, 3, figsize=(20, 6))

In [ ]:
        fig.subplots_adjust(hspace=0.6, wspace=0.4)

In [ ]:
        for j, param in enumerate(results.keys()):

In [ ]:
            plot_sensitivity_analysis(results, time_array, param, axs[j], method)

In [ ]:
        overall_title = f'Sensitivity Analysis for {basin} - {formation} ({method})'

In [ ]:
        fig.suptitle(overall_title, fontsize=16, fontweight='bold', fontname='Arial')

In [ ]:
        # Create the directory for saving plots if it does not exist

In [ ]:
        basin_formation_dir = os.path.join(output_dir, f"{basin}_{formation}")

In [ ]:
        ensure_directory_exists(basin_formation_dir)

In [ ]:
        # Save the plot

In [ ]:
        plot_filename = os.path.join(basin_formation_dir, f'{method}_sensitivity_analysis.png')

In [ ]:
        plt.savefig(plot_filename, format='png', dpi=300)

In [ ]:
        plt.close()

In [ ]:
# Example usage:

In [ ]:
output_dir = 'sensitivity_analysis_plots'

In [ ]:
time_array = np.linspace(0, 60, 61)  # Example time array for 60 months

In [ ]:
best_performing_indices = find_best_performing_index(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns)

In [ ]:
best_performing_rows = identify_best_performing_rows(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns, best_performing_indices)

In [ ]:
save_combined_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir)

In [ ]:
save_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir)

In [ ]:
# def identify_best_performing_rows(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns):

In [ ]:
#     best_performing_rows = {}

In [ ]:
#     for (basin, formation, config_str), model in models.items():

In [ ]:
#         combo_test = filter_by_basin_and_formation(test_df, basin, formation)

In [ ]:
#         numerical_data = combo_test[numerical_columns].values

In [ ]:
#         categorical_data = [combo_test[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
#         y_true = combo_test[y_headers].values

In [ ]:
#         y_pred = predict_with_model(model, numerical_data, categorical_data)

In [ ]:
#         y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
#         y_true_denormalized = denormalize_data_output(y_true, output_scaler, log_transform_columns)

In [ ]:
#         mse = mean_squared_error(y_true_denormalized, y_pred_denormalized, multioutput='raw_values')

In [ ]:
#         if mse.size > 0:

In [ ]:
#             best_match_idx = mse.mean(axis=0).argmin()

In [ ]:
#             if best_match_idx < len(combo_test):

In [ ]:
#                 best_match_row = combo_test.iloc[best_match_idx]

In [ ]:
#             else:

In [ ]:
#                 continue

In [ ]:
#         else:

In [ ]:
#             continue

In [ ]:
#         best_performing_rows[(basin, formation, config_str)] = {

In [ ]:
#             'model': model,

In [ ]:
#             'best_match_row': best_match_row

In [ ]:
#         }

In [ ]:
#     return best_performing_rows

In [ ]:
# def generate_sensitivity_data(best_performing_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array):

In [ ]:
#     variations = {

In [ ]:
#         'HORIZONTIAL_WELL_LENGTH': np.arange(8000, 11500, 500),

In [ ]:
#         'ProppantPerFoot': np.arange(1000, 2200, 200),

In [ ]:
#         'FluidPerFoot_bblft': np.arange(40, 70, 10)

In [ ]:
#     }

In [ ]:
#     results = {key: [] for key in variations.keys()}

In [ ]:
#     for param, values in variations.items():

In [ ]:
#         for value in values:

In [ ]:
#             modified_row = best_performing_row.copy()

In [ ]:
#             modified_row[param] = value

In [ ]:
#             scaled_numerical = input_scaler.transform(pd.DataFrame([modified_row[numerical_columns]], columns=numerical_columns))

In [ ]:
#             categorical_data = [modified_row[col].astype(int).reshape(-1, 1) for col in categorical_columns]

In [ ]:
#             y_pred = predict_with_model(model, scaled_numerical, categorical_data)

In [ ]:
#             y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
#             # Convert to NumPy array for indexing

In [ ]:
#             y_pred_denormalized = np.array(y_pred_denormalized)

In [ ]:
#             production_rates = modified_hyperbolic(time_array, 

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_InitialProd')],

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_DiCoefficient')],

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_BCoefficient')],

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_LimDeclineRate')],

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_BuildupRate')],

In [ ]:
#                                                    y_pred_denormalized[0, y_headers.index('Oil_Params_P50_MonthsInProd')])

In [ ]:
#             results[param].append((value, production_rates))

In [ ]:
#     return results

In [ ]:
# def plot_sensitivity_analysis(results, time_array, param, ax, method):

In [ ]:
#     for value, production in results[param]:

In [ ]:
#         ax.plot(time_array, production, label=f'{param}={value}')

In [ ]:
#     ax.set_yscale('log')

In [ ]:
#     ax.set_xlabel('Time (Months)', fontsize=12, fontname='Arial')

In [ ]:
#     ax.set_ylabel('Production Rate (bbl/day)', fontsize=12, fontname='Arial')

In [ ]:
#     wrapped_title = "\n".join(wrap(f'Sensitivity Analysis for {param} ({method})', 30))

In [ ]:
#     ax.set_title(wrapped_title, fontsize=14, fontweight='bold', fontname='Arial')

In [ ]:
#     ax.legend()

In [ ]:
#     ax.grid(True, which='both', linestyle='--')

In [ ]:
# def save_combined_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir):

In [ ]:
#     plots_by_combination = {}

In [ ]:
#     for (basin, formation, config_str), info in best_performing_rows.items():

In [ ]:
#         model = info['model']

In [ ]:
#         best_match_row = info['best_match_row']

In [ ]:
#         results = generate_sensitivity_data(best_match_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array)

In [ ]:
#         method = 'Neural Network' if 'embedding_output_dim' in config_str else config_str.split(' ')[0]

In [ ]:
#         if (basin, formation) not in plots_by_combination:

In [ ]:
#             plots_by_combination[(basin, formation)] = {'methods': [], 'results': []}

In [ ]:
#         plots_by_combination[(basin, formation)]['methods'].append(method)

In [ ]:
#         plots_by_combination[(basin, formation)]['results'].append(results)

In [ ]:
#     for (basin, formation), data in plots_by_combination.items():

In [ ]:
#         methods = data['methods']

In [ ]:
#         results_list = data['results']

In [ ]:
#         fig, axs = plt.subplots(len(methods), 3, figsize=(20, 6 * len(methods)))

In [ ]:
#         fig.subplots_adjust(hspace=0.6, wspace=0.4)

In [ ]:
#         for i, (method, results) in enumerate(zip(methods, results_list)):

In [ ]:
#             for j, param in enumerate(results.keys()):

In [ ]:
#                 plot_sensitivity_analysis(results, time_array, param, axs[i, j], method)

In [ ]:
#         overall_title = f'Sensitivity Analysis for {basin} - {formation}'

In [ ]:
#         fig.suptitle(overall_title, fontsize=16, fontweight='bold', fontname='Arial')

In [ ]:
#         basin_formation_dir = os.path.join(output_dir, f"{basin}_{formation}")

In [ ]:
#         ensure_directory_exists(basin_formation_dir)

In [ ]:
#         plot_filename = os.path.join(basin_formation_dir, f'sensitivity_analysis_combined.png')

In [ ]:
#         plt.savefig(plot_filename, format='png', dpi=300)

In [ ]:
#         plt.close()

In [ ]:
# def save_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir):

In [ ]:
#     for (basin, formation, config_str), info in best_performing_rows.items():

In [ ]:
#         model = info['model']

In [ ]:
#         best_match_row = info['best_match_row']

In [ ]:
#         results = generate_sensitivity_data(best_match_row, model, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time)

In [ ]:
#         method = 'Neural Network' if 'embedding_output_dim' in config_str else config_str.split(' ')[0]

In [ ]:
#         fig, axs = plt.subplots(1, 3, figsize=(20, 6))

In [ ]:
#         fig.subplots_adjust(hspace=0.6, wspace=0.4)

In [ ]:
#         for j, param in enumerate(results.keys()):

In [ ]:
#             plot_sensitivity_analysis(results, time_array, param, axs[j], method)

In [ ]:
#         overall_title = f'Sensitivity Analysis for {basin} - {formation} ({method})'

In [ ]:
#         fig.suptitle(overall_title, fontsize=16, fontweight='bold', fontname='Arial')

In [ ]:
#         # Create the directory for saving plots if it does not exist

In [ ]:
#         basin_formation_dir = os.path.join(output_dir, f"{basin}_{formation}")

In [ ]:
#         ensure_directory_exists(basin_formation_dir)

In [ ]:
#         # Save the plot

In [ ]:
#         plot_filename = os.path.join(basin_formation_dir, f'{method}_sensitivity_analysis.png')

In [ ]:
#         plt.savefig(plot_filename, format='png', dpi=300)

In [ ]:
#         plt.close()

In [ ]:
# # Example usage:

In [ ]:
# # Example usage:

In [ ]:
# output_dir = 'sensitivity_analysis_plots'

In [ ]:
# ttime_array = np.linspace(1, 12 * years, 12 * years)  # Example time array for 60 months

In [ ]:
# best_performing_rows = identify_best_performing_rows(models, test_df, numerical_columns, categorical_columns, y_headers, output_scaler, log_transform_columns)

In [ ]:
# save_combined_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir)

In [ ]:
# save_grid_plot(best_performing_rows, numerical_columns, categorical_columns, input_scaler, output_scaler, log_transform_columns, y_headers, time_array, output_dir)

In [ ]:
# Defining a dictionary to store all outputs

In [ ]:
outputs = {

In [ ]:
    'train_df': train_df,

In [ ]:
    'val_df': val_df,

In [ ]:
    'test_df': test_df,

In [ ]:
    'models': models,

In [ ]:
    #'shap_values_dict': shap_values_dict,

In [ ]:
    'evaluation_results': evaluation_results,

In [ ]:
    'best_performing_models': best_performing_models,

In [ ]:
    'task_times': task_times,

In [ ]:
    'y_headers': y_headers,

In [ ]:
    'numerical_columns': numerical_columns,

In [ ]:
    'categorical_columns': categorical_columns

In [ ]:
}

In [ ]:
# Saving the dictionary to a pickle file

In [ ]:
output_file_path = r'C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\outputs.pkl'

In [ ]:
with open(output_file_path, 'wb') as file:

In [ ]:
    pickle.dump(outputs, file)

In [ ]:
output_file_path

In [ ]:
# Read the first Excel sheet (cumulative data)

In [ ]:
file_path = r'C:\Users\Prakhar.Sarkar\OneDrive - SRP Management Services\Documents\_For_Prakhar\TCTest.xlsx'

In [ ]:
Test_df = pd.read_excel(file_path)

In [ ]:
Test_df

In [ ]:
# Columns to be parsed

In [ ]:
param_columns = [

In [ ]:
    'Oil_Params_P50', 'Gas_Params_P50', 'Water_Params_P50'

In [ ]:
]

In [ ]:
# Apply robust parsing to each column

In [ ]:
for col in param_columns:

In [ ]:
    Test_df[col] = Test_df[col].apply(robust_parse)

In [ ]:
# Split each parameter into its own column

In [ ]:
new_columns = []

In [ ]:
for col in param_columns:

In [ ]:
    expanded_cols = [f'{col}_Method', f'{col}_BuildupRate', f'{col}_MonthsInProd', 

In [ ]:
                     f'{col}_InitialProd', f'{col}_DiCoefficient', f'{col}_BCoefficient', 

In [ ]:
                     f'{col}_LimDeclineRate']

In [ ]:
    temp_df = pd.DataFrame(Test_df[col].tolist(), columns=expanded_cols)

In [ ]:
    # Convert numeric columns to float

In [ ]:
    for num_col in expanded_cols:

In [ ]:
        if num_col.endswith('_Method'):

In [ ]:
            temp_df[num_col] = temp_df[num_col].astype(str)  # Ensuring 'Method' is of string type

In [ ]:
        else:

In [ ]:
            temp_df[num_col] = pd.to_numeric(temp_df[num_col], errors='coerce')  # Convert to numeric and handle errors

In [ ]:
    Test_df = pd.concat([Test_df, temp_df], axis=1)

In [ ]:
# Drop the original parameter columns

In [ ]:
Test_df.drop(columns=param_columns, inplace=True)

In [ ]:
def preprocess_test_data(test_df, numerical_columns, categorical_columns, input_scaler):

In [ ]:
    numerical_data = test_df[numerical_columns].values

In [ ]:
    scaled_numerical = input_scaler.transform(numerical_data)

In [ ]:
    categorical_data = [test_df[col].astype(int).values.reshape(-1, 1) for col in categorical_columns]

In [ ]:
    return scaled_numerical, categorical_data

In [ ]:
# Assuming test_df is already defined

In [ ]:
scaled_numerical, categorical_data = preprocess_test_data(Test_df, numerical_columns, categorical_columns, input_scaler)

In [ ]:
def generate_predictions(models, scaled_numerical, categorical_data, output_scaler, log_transform_columns):

In [ ]:
    predictions = {}

In [ ]:
    for (basin, formation, config_str), model in models.items():

In [ ]:
        y_pred = predict_with_model(model, scaled_numerical, categorical_data)

In [ ]:
        y_pred_denormalized = denormalize_data_output(y_pred, output_scaler, log_transform_columns)

In [ ]:
        predictions[(basin, formation, config_str)] = y_pred_denormalized

In [ ]:
    return predictions

In [ ]:
predictions = generate_predictions(models, scaled_numerical, categorical_data, output_scaler, log_transform_columns)

In [ ]:
def generate_production_rates(y_pred_denormalized, y_true_denormalized, headers, time):

In [ ]:
    predicted_productions = []

In [ ]:
    actual_productions = []

In [ ]:
    for idx in range(len(y_pred_denormalized)):

In [ ]:
        qi_pred = y_pred_denormalized[idx, headers.index('Oil_Params_P50_InitialProd')]

In [ ]:
        di_pred = y_pred_denormalized[idx, headers.index('Oil_Params_P50_DiCoefficient')]

In [ ]:
        b_pred  = y_pred_denormalized[idx, headers.index('Oil_Params_P50_BCoefficient')]

In [ ]:
        IBU_pred = y_pred_denormalized[idx, headers.index('Oil_Params_P50_BuildupRate')]

In [ ]:
        MBU_pred = y_pred_denormalized[idx, headers.index('Oil_Params_P50_MonthsInProd')]

In [ ]:
        Dlim_pred = y_pred_denormalized[idx, headers.index('Oil_Params_P50_LimDeclineRate')]

In [ ]:
        qi_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_InitialProd')]

In [ ]:
        di_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_DiCoefficient')]

In [ ]:
        b_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_BCoefficient')]

In [ ]:
        IBU_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_BuildupRate')]

In [ ]:
        MBU_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_MonthsInProd')]

In [ ]:
        Dlim_true = y_true_denormalized[idx, headers.index('Oil_Params_P50_LimDeclineRate')]

In [ ]:
        predicted_production = modified_hyperbolic(time, qi_pred, di_pred, b_pred, Dlim_pred, IBU_pred, MBU_pred)

In [ ]:
        actual_production = modified_hyperbolic(time, qi_true, di_true, b_true, Dlim_true, IBU_true, MBU_true)

In [ ]:
        predicted_productions.append(predicted_production)

In [ ]:
        actual_productions.append(actual_production)

In [ ]:
    return predicted_productions, actual_productions

In [ ]:
def plot_comparison(test_df, predictions, y_headers, time):

In [ ]:
    for (basin, formation, config_str), y_pred in predictions.items():

In [ ]:
        y_true = test_df[y_headers].values

In [ ]:
        y_true_denormalized = denormalize_data_output(y_true, output_scaler, log_transform_columns).values

In [ ]:
        # Generate production rates

In [ ]:
        predicted_productions, actual_productions = generate_production_rates(y_pred.values, y_true_denormalized, y_headers, time)

In [ ]:
        for idx in range(len(predicted_productions)):

In [ ]:
            plot_decline_curves(time, actual_productions[idx], predicted_productions[idx], f'Predicted vs Actual Decline Curve for {basin} - {formation} - {config_str} - Point {idx+1}')

In [ ]:
# Example usage

In [ ]:
years = 5

In [ ]:
time = np.linspace(1, 12 * years, 12 * years)  # 5 Years

In [ ]:
plot_comparison(Test_df, predictions, y_headers, time)